# EFGP / Active-Box 全实验 Colab 总控（10M–300M）

本 notebook 把原有实验和新增实验放在同一入口中，但保留三条**不可混表计时**的证据链：

| protocol | 内容 | 可以说明什么 |
|---|---|---|
| `archived_complete_pipeline` | 原 `group_a/group_b/group_c`，direct CG 与 binned-C1 candidates，含 setup/solve/prediction | 原论文探索性规模图与候选筛选 |
| `controlled_fixed_system` | 所有方法共享同一个哈希 (A\beta=b)，1 warm-up + 5 paired repeats | 方法间严格配对加速、构造/求解/存储权衡 |
| `prediction_audit` | 读取 timed system 与保存的 canonical timed \(\beta\)，仅计算 test RMSE | 同一计时解的预测等价性；其中 prediction 时间不进入 speedup claim |

使用方式：选择 Colab 的 **A100 GPU + High-RAM** runtime，然后点击 **运行时 → 全部运行**。默认的 `RUN_ALL_FORMAL_EXPERIMENTS=True` 会自动完成 smoke、10M 主实验、prediction audit、OAT、box-budget 消融、Synthetic nested-prefix 10M–300M，以及 Winnebago archived-exact 10M–300M。每个规模是独立 job；失败会结构化记录并隔离，不会用 `CalledProcessError` 阻断后续实验。

参考并保留了 `boxeig_inverse_diagnostics_experiment.ipynb` 中最关键的 direct/binned precompute policy；统一结果区重新整理 legacy、controlled 和 audit 三种 schema。


In [ ]:
# ==================== 一键正式实验：通常无需修改 ====================
from pathlib import Path

REPO_URL = "https://github.com/Yifiwifi/EFGP-Eigenpro.git"
REPO_REF = "codex/colab-all-experiments"
LOCAL_REPO = Path("/content/EFGP-Eigenpro")

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/EFGP_Colab")
DRIVE_DATA_ROOT = DRIVE_PROJECT_ROOT / "data_bundle"
RUN_TAG_PREFIX = "paper_one_click"
# checkout 后自动使用 paper_one_click_<git sha>，同一代码自动 resume，
# 新代码自动进入新目录，不会覆盖旧证据。
RUN_TAG = None
DRIVE_RUN_ROOT = None
LOCAL_DATA_DIR = Path("/content/efgp_data")

RUN_ALL_FORMAL_EXPERIMENTS = True
FORMAL_SCALE_SIZES = [10_000_000, 30_000_000, 100_000_000, 300_000_000]

# 从 drive_manifest.json 自动选择正式 campaign 所需数据。
DATA_BUNDLES = []
CACHE_DATA_LOCALLY = True       # 正式计时推荐 True，避免 Drive FUSE 进入 timing
VERIFY_FULL_SHA256 = False      # 只跳过无关 bundle；本次选中 artifact 仍逐个验 SHA-256

# 一键正式 campaign 不重跑不可直接混表的 legacy exploratory groups。
RUN_LEGACY_GROUPS = []

# 下列变量由一键模式统一设定；False 时仍可作为 advanced/manual 模式使用。
RUN_PLUMBING_SMOKE = RUN_ALL_FORMAL_EXPERIMENTS
RUN_CG_SCREEN_10M = False
RUN_Q256_CENTER_10M = RUN_ALL_FORMAL_EXPERIMENTS
RUN_BOX_BUDGET_ABLATION = RUN_ALL_FORMAL_EXPERIMENTS
RUN_ARCHIVED_EXACT_SCALE = RUN_ALL_FORMAL_EXPERIMENTS
RUN_DEVELOPMENT_MASTER_SCALE = RUN_ALL_FORMAL_EXPERIMENTS
RUN_MANITOWOC_SCALE = False
RUN_WINNEBAGO_OAT_10M = RUN_ALL_FORMAL_EXPERIMENTS
RUN_Q128_BRIDGE = False
RUN_SE_FULL_INVERSE_CONTROL = False
RUN_PREDICTION_AUDIT = RUN_ALL_FORMAL_EXPERIMENTS

ACTIVE_SIZES = list(FORMAL_SCALE_SIZES) if RUN_ALL_FORMAL_EXPERIMENTS else [10_000_000]
ALLOW_100M = RUN_ALL_FORMAL_EXPERIMENTS
ALLOW_300M = RUN_ALL_FORMAL_EXPERIMENTS
PREDICTION_AUDIT_MAX_TEST_N = 2_500_000
PREDICTION_AUDIT_PROFILES = ["paper_10m", "scale_development_masters"]

# 一键模式中的 profile 级数据族过滤。尤其禁止把已知失败的
# Winnebago raw-prefix development cases 混入 Synthetic scale job。
PROFILE_DATASET_FAMILIES = {
    "scale_development_masters": ["Synthetic"],
    "scale_archived_exact": ["Winnebago"],
} if RUN_ALL_FORMAL_EXPERIMENTS else {}
ACTIVE_CASE_IDS = []

# 缺失数据的可选生成动作；默认不执行。
# controlled scale 缺 30/100/300M；完整 legacy groups 还需补 1/3M。
GENERATE_ARCHIVED_SYNTHETIC_SIZES = []  # noise=.3 / chunk=5M / _ntrainN
GENERATE_MANITOWOC_300M = False
MANITOWOC_START_LOD = 8  # 只是起点；容量不足时必须提高并重新精确扫描
SYNC_GENERATED_DATA_TO_DRIVE = False

required_bundles = []
if RUN_LEGACY_GROUPS:
    required_bundles.append("legacy_named_route_inputs")
if RUN_ARCHIVED_EXACT_SCALE:
    required_bundles.append("archived_exact_available")
if RUN_DEVELOPMENT_MASTER_SCALE:
    required_bundles.append("development_scale_masters")
if RUN_MANITOWOC_SCALE:
    required_bundles.append("manitowoc_10m")
if any([
    RUN_CG_SCREEN_10M, RUN_Q256_CENTER_10M, RUN_BOX_BUDGET_ABLATION,
    RUN_WINNEBAGO_OAT_10M,
    RUN_Q128_BRIDGE, RUN_SE_FULL_INVERSE_CONTROL,
]):
    required_bundles.append("controlled_10m")
DATA_BUNDLES = list(dict.fromkeys([*DATA_BUNDLES, *required_bundles]))

requested_size_hints = [int(n) for n in ACTIVE_SIZES]
if any(group in {"group_b", "group_c"} for group in RUN_LEGACY_GROUPS):
    requested_size_hints.append(300_000_000)
elif "group_a" in RUN_LEGACY_GROUPS:
    requested_size_hints.append(30_000_000)
requested_size_hints.extend(int(n) for n in GENERATE_ARCHIVED_SYNTHETIC_SIZES)
if GENERATE_MANITOWOC_300M:
    requested_size_hints.append(300_000_000)
if RUN_PREDICTION_AUDIT:
    requested_size_hints.extend([10_000_000, 100_000_000, 300_000_000])
MAX_REQUESTED_N = max(requested_size_hints or [0])

DISCONNECT_RUNTIME_WHEN_VERIFIED = False
print("Configuration loaded. No heavy experiment has started.")


## 1. 挂载 Drive、固定输出目录

数据与结果分开：Drive 长期保存 master/manifest 和 run artifacts；当前 case 的数据先复制到 `/content` 本地 SSD 后再计时。


In [ ]:
import os, sys, json, shutil, subprocess, time, hashlib, platform
from pathlib import Path
from IPython.display import display

IS_COLAB = "google.colab" in sys.modules
NOTEBOOK_STARTED_UTC = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
NOTEBOOK_STARTED_PERF_COUNTER = time.perf_counter()
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Not running in Colab; Drive mount skipped.")

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Drive project:", DRIVE_PROJECT_ROOT)
print("Local data cache:", LOCAL_DATA_DIR)


## 2. Clone/pin 代码并安装 Colab 依赖

正式论文运行必须把 `REPO_REF` 固定到 commit SHA，并在最终 run manifest 中记录。不要同时安装多个 CuPy wheel；本格先检查已有 CuPy，再补齐 cuFINUFFT 和通用依赖。当前内部 runner 不需要额外安装 EigenPro2/3。


In [ ]:
def run_cmd(args, *, cwd=None, env=None, check=True):
    args = [str(x) for x in args]
    print("+", " ".join(args))
    return subprocess.run(args, cwd=cwd, env=env, check=check)

if not LOCAL_REPO.exists():
    run_cmd(["git", "clone", REPO_URL, str(LOCAL_REPO)])
run_cmd(["git", "fetch", "--all", "--tags"], cwd=LOCAL_REPO)
repo_ref_text = str(REPO_REF).strip()
is_full_sha = len(repo_ref_text) == 40 and all(
    char in "0123456789abcdefABCDEF" for char in repo_ref_text
)
checkout_ref = repo_ref_text if is_full_sha else f"origin/{repo_ref_text}"
run_cmd(["git", "checkout", "--detach", checkout_ref], cwd=LOCAL_REPO)

run_cmd([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
installed = json.loads(subprocess.check_output(
    [sys.executable, "-m", "pip", "list", "--format=json"], text=True
))
cupy_distributions = sorted(
    row["name"] for row in installed
    if row["name"].lower() == "cupy" or row["name"].lower().startswith("cupy-cuda")
)
if len(cupy_distributions) > 1:
    raise RuntimeError(f"Multiple CuPy distributions are installed: {cupy_distributions}")
if not cupy_distributions:
    run_cmd([sys.executable, "-m", "pip", "install", "cupy-cuda12x"])

colab_dependencies = [
    sys.executable, "-m", "pip", "install",
    "numpy>=1.23,<3", "scipy>=1.10,<2", "finufft>=2.3,<3",
    "psutil>=5.9,<8", "pandas", "matplotlib", "cufinufft>=2.4,<3",
]
if GENERATE_MANITOWOC_300M:
    colab_dependencies.extend(["pyproj>=3.6,<4", "laspy", "lazrs"])
run_cmd(colab_dependencies)

import cupy as cp
print("CuPy distribution:", cupy_distributions or ["cupy-cuda12x (installed above)"])
print("CuPy version:", cp.__version__)

os.chdir(LOCAL_REPO)
if str(LOCAL_REPO) not in sys.path:
    sys.path.insert(0, str(LOCAL_REPO))
GIT_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=LOCAL_REPO, text=True
).strip()
RUN_TAG = f"{RUN_TAG_PREFIX}_{GIT_SHA[:12]}"
DRIVE_RUN_ROOT = DRIVE_PROJECT_ROOT / "runs" / RUN_TAG
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Pinned Git SHA:", GIT_SHA)
print("Automatic run tag:", RUN_TAG)
print("Run output root:", DRIVE_RUN_ROOT)


## 3. GPU / RAM / 磁盘 preflight

当前 controlled loader 的训练 (x,y) 会转成 float64，最低常驻量约为 (24N) bytes。300M 仅训练数组就约 6.71 GiB GPU；建议 A100 40/80GB 且可用 host RAM ≥20GB。L4 的 300M 只能先试 `cg/default/full-eig`，不能预先承诺全六方法成功。


In [ ]:
import numpy as np, pandas as pd, psutil
import cupy as cp
import scipy, cufinufft, finufft

run_cmd(["nvidia-smi"])
props = cp.cuda.runtime.getDeviceProperties(0)
gpu_name = props["name"].decode() if isinstance(props["name"], bytes) else str(props["name"])
gpu_total = int(cp.cuda.runtime.memGetInfo()[1])
host_available = int(psutil.virtual_memory().available)
disk_free = int(shutil.disk_usage("/content").free) if Path("/content").exists() else int(shutil.disk_usage(".").free)
runtime_info = {
    "git_sha": GIT_SHA,
    "gpu": gpu_name,
    "gpu_total_bytes": gpu_total,
    "host_available_bytes": host_available,
    "local_disk_free_bytes": disk_free,
    "python": sys.version,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "cupy": cp.__version__,
    "cufinufft": getattr(cufinufft, "__version__", "unknown"),
    "finufft": getattr(finufft, "__version__", "unknown"),
}
display(pd.DataFrame([
    {
        "N": n,
        "train_x_y_fp64_GiB": 24*n/2**30,
        "host_load_peak_est_GiB": 28*n/2**30,
        "logical_test_rows": n//4,
    }
    for n in [10_000_000, 30_000_000, 100_000_000, 300_000_000]
]))
print(json.dumps(runtime_info, indent=2))

CAN_RUN_300M = bool(
    gpu_total >= 30 * 2**30 and host_available >= 20 * 2**30
)
if MAX_REQUESTED_N >= 100_000_000 and not ALLOW_100M:
    raise RuntimeError("The selected workload reaches >=100M; set ALLOW_100M=True after reviewing memory.")
if MAX_REQUESTED_N >= 300_000_000:
    if not ALLOW_300M:
        raise RuntimeError("300M is gated; set ALLOW_300M=True explicitly.")
    if not CAN_RUN_300M and not RUN_ALL_FORMAL_EXPERIMENTS:
        raise RuntimeError("300M requires >=30 GiB GPU and >=20 GiB currently available host RAM.")
    if not CAN_RUN_300M:
        print(
            "WARNING: 300M jobs will be marked SKIPPED_HARDWARE; "
            "10M/30M/100M jobs will still run."
        )


## 4. Drive manifest 校验与单份 master staging

`drive_manifest.json` 由 `colab_drive_pack.py` 生成。开发用 300M Synthetic（noise=.02）和 Winnebago raw master 只属于新的 `controlled_master_prefix` protocol，不能冒充论文 archived Synthetic（noise=.3）或原 spatial-stratified exact artifacts。

对 ZIP_STORED master，controlled runner 的 `subset_mode='prefix'` 会直接 memory-map 所需的 10M/30M/100M 前缀；不会先加载全部 300M。10M 正式输入登记到 `controlled_10m`；旧 named routes 的现有输入登记到 `legacy_named_route_inputs`，但 notebook 仍会强制检查完整 noise=.3 Synthetic `_ntrainN`，不会接受 low-noise fallback。


In [ ]:
from efgp_eigenpro_py.gpu.benchmark_dataset.colab_drive_pack import (
    compare_nested_prefix,
    inspect_stored_npz,
    verify_catalog,
)

DATA_MANIFEST = DRIVE_DATA_ROOT / "drive_manifest.json"
if not DATA_MANIFEST.is_file():
    raise FileNotFoundError(
        f"Missing {DATA_MANIFEST}. Build/upload the catalog using COLAB_DRIVE_DATA.md first."
    )
catalog = json.loads(DATA_MANIFEST.read_text(encoding="utf-8"))
DATA_MANIFEST_SHA256 = hashlib.sha256(DATA_MANIFEST.read_bytes()).hexdigest()
DATA_MANIFEST_SNAPSHOT = DRIVE_RUN_ROOT / "data_manifest_snapshot.json"
if DATA_MANIFEST_SNAPSHOT.is_file():
    snapshot_sha256 = hashlib.sha256(DATA_MANIFEST_SNAPSHOT.read_bytes()).hexdigest()
    if snapshot_sha256 != DATA_MANIFEST_SHA256:
        raise RuntimeError(
            "The run directory already contains a different data manifest snapshot; "
            "use a new RUN_TAG instead of mixing data catalogs."
        )
else:
    snapshot_partial = DATA_MANIFEST_SNAPSHOT.with_suffix(".json.partial")
    shutil.copy2(DATA_MANIFEST, snapshot_partial)
    snapshot_partial.replace(DATA_MANIFEST_SNAPSHOT)
available_bundles = sorted(catalog.get("bundles", {}))
print("Available bundles:", available_bundles)
missing_bundles = [name for name in DATA_BUNDLES if name not in catalog.get("bundles", {})]
if missing_bundles:
    raise KeyError(f"Unknown DATA_BUNDLES={missing_bundles}; available={available_bundles}")

selected_names = sorted({
    artifact_name
    for bundle in DATA_BUNDLES
    for artifact_name in catalog["bundles"][bundle]
})
artifacts_by_name = {row["name"]: row for row in catalog.get("artifacts", [])}
selected_artifacts = [artifacts_by_name[name] for name in selected_names]

if VERIFY_FULL_SHA256:
    print("Full SHA-256 verification of the complete catalog...")
    verify_catalog(DATA_MANIFEST)

def streaming_sha256(path, chunk_bytes=16 * 2**20):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(int(chunk_bytes))
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

# Fail before copying if two selected catalog entries would collide
# at the flat local-cache basename with different bytes.
selected_by_basename = {}
for row in selected_artifacts:
    source = DRIVE_DATA_ROOT / row["relative_path"]
    basename = source.name
    prior = selected_by_basename.get(basename)
    if prior is not None and prior["sha256"] != row["sha256"]:
        raise RuntimeError(
            f"Selected artifacts collide at {basename!r} with different SHA-256 values."
        )
    selected_by_basename[basename] = row

cache_valid_by_basename = {}
additional_cache_bytes = 0
for basename, row in selected_by_basename.items():
    source = DRIVE_DATA_ROOT / row["relative_path"]
    if not source.is_file():
        raise FileNotFoundError(source)
    expected_size = int(row["size_bytes"])
    if source.stat().st_size != expected_size:
        raise ValueError(f"Byte-size mismatch: {source}")
    destination = LOCAL_DATA_DIR / basename
    cache_valid = False
    if CACHE_DATA_LOCALLY and destination.is_file() and destination.stat().st_size == expected_size:
        print("Verifying selected local cache SHA-256:", basename)
        cache_valid = streaming_sha256(destination) == row["sha256"]
    cache_valid_by_basename[basename] = cache_valid
    if CACHE_DATA_LOCALLY and not cache_valid:
        existing_size = destination.stat().st_size if destination.is_file() else 0
        additional_cache_bytes += max(expected_size - int(existing_size), 0)
if CACHE_DATA_LOCALLY and additional_cache_bytes > shutil.disk_usage("/content").free:
    raise RuntimeError(
        "Selected uncached bytes do not fit on the current Colab local disk."
    )

staged = {}
inspected_masters = set()
for row in selected_artifacts:
    source = DRIVE_DATA_ROOT / row["relative_path"]
    destination = LOCAL_DATA_DIR / source.name
    if CACHE_DATA_LOCALLY:
        if not cache_valid_by_basename[source.name]:
            print("Copying and SHA-verifying selected artifact:", source.name)
            digest = hashlib.sha256()
            with source.open("rb") as source_handle, destination.open("wb") as destination_handle:
                while True:
                    block = source_handle.read(16 * 2**20)
                    if not block:
                        break
                    digest.update(block)
                    destination_handle.write(block)
            if digest.hexdigest() != row["sha256"]:
                raise ValueError(f"Catalog SHA-256 mismatch while copying: {source}")
            if destination.stat().st_size != int(row["size_bytes"]):
                raise ValueError(f"Copied byte-size mismatch: {destination}")
            cache_valid_by_basename[source.name] = True
    else:
        destination = source
        print("SHA-verifying selected Drive artifact:", source.name)
        if streaming_sha256(destination) != row["sha256"]:
            raise ValueError(f"Catalog SHA-256 mismatch: {source}")
        # The runners always receive LOCAL_DATA_DIR.  In no-cache
        # mode expose the verified Drive file there through a
        # symlink instead of silently leaving the dataset absent.
        local_link = LOCAL_DATA_DIR / source.name
        if not (
            local_link.is_symlink()
            and local_link.resolve() == source.resolve()
        ):
            if local_link.exists() or local_link.is_symlink():
                local_link.unlink()
            local_link.symlink_to(source)
        destination = local_link
    staged[row["name"]] = destination
    if row["role"] == "master_npz" and destination not in inspected_masters:
        inspect_stored_npz(destination)
        inspected_masters.add(destination)

# Runner 要求 metadata 与 NPZ 同 stem；若 catalog 保存的是非规范文件名，建立本地规范别名。
for dataset_id, dataset in catalog.get("datasets", {}).items():
    names = set(dataset.get("artifact_names", []))
    if not names.intersection(selected_names):
        continue
    master_name = f"{dataset_id}:master"
    metadata_name = f"{dataset_id}:metadata"
    if master_name in staged and metadata_name in staged:
        canonical_json = LOCAL_DATA_DIR / (staged[master_name].stem + ".json")
        metadata_sha256 = streaming_sha256(staged[metadata_name])
        if (
            not canonical_json.is_file()
            or streaming_sha256(canonical_json) != metadata_sha256
        ):
            shutil.copy2(staged[metadata_name], canonical_json)

REPO_PROCESSED = LOCAL_REPO / "efgp_eigenpro_py/gpu/benchmark_dataset/processed"
REPO_PROCESSED.mkdir(parents=True, exist_ok=True)
for local_file in LOCAL_DATA_DIR.iterdir():
    if local_file.suffix.lower() not in {".npz", ".json"}:
        continue
    link = REPO_PROCESSED / local_file.name
    if not link.exists():
        link.symlink_to(local_file)

os.environ["BTAB_PROCESSED_DIR"] = str(LOCAL_DATA_DIR)
display(pd.DataFrame([
    {"artifact": row["name"], "role": row["role"], "GiB": int(row["size_bytes"])/2**30}
    for row in selected_artifacts
]))


## 5. 可选：补建缺失的 archived Synthetic / Manitowoc master

archived Synthetic 必须是 `_ntrainN`、noise=0.3、train/test seeds 20260421/1、generation chunk=5M。现有 `_nN` 300M master 是 noise=.02 development artifact，不能重命名代替。

Manitowoc 300M 必须保持冻结 AOI、hash split、浅层优先顺序。LOD 8 只是起点；若精确容量不足 300M/75M，应提高到 LOD 9，不能用密度估算替代扫描结果。


In [ ]:
generated_before = {path.resolve() for path in LOCAL_DATA_DIR.iterdir()}
if GENERATE_ARCHIVED_SYNTHETIC_SIZES:
    sizes_arg = ",".join(str(int(n)) for n in GENERATE_ARCHIVED_SYNTHETIC_SIZES)
    run_cmd([
        sys.executable, "-m",
        "efgp_eigenpro_py.gpu.benchmark_dataset.preprocess_synthetic_true_func_2d_size_sweep",
        "--n-train-list", sizes_arg,
        "--noise", "0.3",
        "--seed-train", "20260421",
        "--seed-test", "1",
        "--chunk-rows", "5000000",
        "--size-token", "ntrain",
        "--output-dir", str(LOCAL_DATA_DIR),
    ], cwd=LOCAL_REPO)

if GENERATE_MANITOWOC_300M:
    run_cmd([
        sys.executable, "-m",
        "efgp_eigenpro_py.gpu.benchmark_dataset.preprocess_usgs_ept_wi_2county",
        "--ept-json", "https://s3-us-west-2.amazonaws.com/usgs-lidar-public/WI_2County_1_B23/ept.json",
        "--n-train-list", "300000000", "--allow-large-output",
        "--aoi-center-x", "437100", "--aoi-center-y", "4884400", "--aoi-side-m", "52000",
        "--aoi-crs", "EPSG:6345", "--max-lod-depth", str(MANITOWOC_START_LOD),
        "--source-project", "WI_2County_1_B23",
        "--official-mean-ground-density", "11.37",
        "--official-work-unit-area-km2", "1660.1823787253759",
        "--cache-dir", str(DRIVE_PROJECT_ROOT / "ept_cache/WI_2County_1_B23"),
        "--temporary-dir", str(LOCAL_DATA_DIR / "_manitowoc_300m_work"),
        "--output-dir", str(LOCAL_DATA_DIR),
        "--dataset-stem-prefix", "USGS_EPT_WI_2County_1_B23_full_workunit_ground_elevation",
    ], cwd=LOCAL_REPO)
    manitowoc_master = LOCAL_DATA_DIR / "USGS_EPT_WI_2County_1_B23_full_workunit_ground_elevation_n300000000.npz"
    manitowoc_frozen_10m = LOCAL_DATA_DIR / "USGS_EPT_WI_2County_1_B23_full_workunit_ground_elevation_n10000000.npz"
    if not manitowoc_master.is_file():
        raise RuntimeError("EPT build did not produce the requested 300M master.")
    prefix_report = compare_nested_prefix(
        larger_npz=manitowoc_master,
        prefix_npz=manitowoc_frozen_10m,
        chunk_rows=1_000_000,
    )
    print("Frozen Manitowoc 10M prefix verified:", prefix_report)

# Make newly generated artifacts visible to the legacy hard-coded processed path.
for generated_file in LOCAL_DATA_DIR.iterdir():
    if generated_file.suffix.lower() not in {".npz", ".json"}:
        continue
    link = REPO_PROCESSED / generated_file.name
    if not link.exists():
        link.symlink_to(generated_file)

if SYNC_GENERATED_DATA_TO_DRIVE:
    generated_drive_dir = DRIVE_DATA_ROOT / "generated_pending_catalog"
    generated_drive_dir.mkdir(parents=True, exist_ok=True)
    generated_now = [
        path for path in LOCAL_DATA_DIR.iterdir()
        if path.resolve() not in generated_before
    ]
    for generated_file in generated_now:
        if generated_file.suffix.lower() not in {".npz", ".json"}:
            continue
        target = generated_drive_dir / generated_file.name
        if not target.exists() or target.stat().st_size != generated_file.stat().st_size:
            print("Syncing generated artifact to Drive:", generated_file.name)
            shutil.copy2(generated_file, target)
    print("Register verified generated files with colab_drive_pack before formal runs.")


## 6. 小规模 GPU plumbing smoke

这不是论文结果。它只验证 cuFINUFFT、GPU eigensolver、固定系统哈希和输出写入；controlled protocol 仍保持 1 warm-up + 5 measured repeats。


In [ ]:
SMOKE_OK = True
SMOKE_RETURN_CODE = None
SMOKE_ELAPSED_SECONDS = None
if RUN_PLUMBING_SMOKE:
    smoke_started = time.perf_counter()
    smoke_stem = "USGS_EPT_WI_2County_1_B23_full_workunit_ground_elevation_n10000000"
    smoke_out = DRIVE_RUN_ROOT / "smoke_cufinufft"
    smoke_result = run_cmd([
        sys.executable, "-m",
        "efgp_eigenpro_py.gpu.box_toeplitz_active_block.controlled.benchmark",
        "--dataset-stem", smoke_stem, "--dataset-dir", str(LOCAL_DATA_DIR),
        "--n-train", "5000", "--subset-mode", "prefix",
        "--kernel", "matern", "--lengthscale", "0.1", "--nu", "1.5",
        "--lambda", "0.1", "--fourier-eps", "1e-3", "--tol", "1e-7",
        "--maxiter", "6000", "--methods", "cg,default",
        "--box-budget", "1024", "--rank", "32",
        "--warmup-repeats", "1", "--measured-repeats", "5",
        "--nufft-backend", "cufinufft", "--strict-gpu-eig",
        "--output-dir", str(smoke_out),
    ], cwd=LOCAL_REPO, check=False)
    SMOKE_RETURN_CODE = int(smoke_result.returncode)
    SMOKE_ELAPSED_SECONDS = time.perf_counter() - smoke_started
    SMOKE_OK = SMOKE_RETURN_CODE == 0
    if not SMOKE_OK:
        print(
            f"SMOKE FAILED (return code {SMOKE_RETURN_CODE}). "
            "Heavy formal jobs will be recorded as SKIPPED_SMOKE_FAILED."
        )


# A. 原有 archived complete-pipeline 实验

这里复用参考 notebook 的 direct-CG / binned-C1 precompute policy。`group_a/b/c` 是 exploratory candidate scans：不同 setup route、单次配置，且并不保证所有方法来自完全相同的 (A,b)。结果必须标记 `archived_complete_pipeline`，不能与 controlled paired speedup 排在同一列。

Legacy runner 没有 case 内 resume。本 notebook 将每个 group 放在独立输出目录；只有最终原子写入 `_SUCCESS.json` 的 group 才会跳过，中断时即使已有部分 CSV 也会重跑该 group。


In [ ]:
from dataclasses import replace
import contextlib, importlib
from efgp_eigenpro_py.gpu.backends import BackendConfig
from efgp_eigenpro_py.gpu.box_toeplitz_active_block.config import BTABExperimentConfig

PACKAGE = "efgp_eigenpro_py.gpu.box_toeplitz_active_block"
MODULE_NAMES = [
    "efgp_eigenpro_py.gpu.versions",
    f"{PACKAGE}.config",
    f"{PACKAGE}.run_experiments",
]

def reload_legacy_modules():
    modules = {}
    for name in MODULE_NAMES:
        modules[name] = importlib.reload(sys.modules[name]) if name in sys.modules else importlib.import_module(name)
    return modules

def validate_archived_synthetic_inputs(required_sizes):
    failures = []
    for n_train in sorted({int(value) for value in required_sizes}):
        stem = f"synthetic_true_func_2d_ntrain{n_train}"
        npz_path = LOCAL_DATA_DIR / f"{stem}.npz"
        json_path = LOCAL_DATA_DIR / f"{stem}.json"
        if not npz_path.is_file() or not json_path.is_file():
            failures.append(f"{stem}: missing NPZ/JSON")
            continue
        metadata = json.loads(json_path.read_text(encoding="utf-8"))
        generation = metadata.get("generation", {})
        expected = {
            "noise_std": 0.3,
            "seed_train": 20260421,
            "seed_test": 1,
            "chunk_rows": 5_000_000,
        }
        mismatches = {
            key: (generation.get(key), value)
            for key, value in expected.items()
            if generation.get(key) != value
        }
        if mismatches:
            failures.append(f"{stem}: archived generation mismatch {mismatches}")
    if failures:
        raise RuntimeError(
            "Archived Synthetic inputs are incomplete or incompatible. "
            "Use GENERATE_ARCHIVED_SYNTHETIC_SIZES first:\n- "
            + "\n- ".join(failures)
        )


In [ ]:
PRECOMPUTE_POLICY = {
    'plain_cg': 'original',
    'eigenpro_pcg': 'c1',
    'btab_inverse': 'c1',
    'btab_boxeig': 'c1',
}

def install_notebook_precompute_policy(reloaded_mods: dict[str, object]) -> dict[str, str]:
    versions_mod = reloaded_mods['efgp_eigenpro_py.gpu.versions']
    run_mod = reloaded_mods[f'{PACKAGE}.run_experiments']

    bench_mod = importlib.import_module(
        'efgp_eigenpro_py.gpu.benchmark_dataset.accuracy_based_eigenpro_tables'
    )
    bench_mod = importlib.reload(bench_mod)
    bench_cfg = bench_mod.AccuracyBenchmarkConfig(
        precompute_methods={'default': 'c1'},
        binned_quality='balanced',
        binned_use_sparse_bins=False,
        binned_use_gpu_dense_bins=True,
        binned_allow_exact_nufft_fallback=False,
        binned_nufft_allow_cpu_fallback=False,
    )
    bench_mod.install_gpu_precompute_patch(bench_cfg)
    patched_gpu_precompute = bench_mod._gpu_v1_ops_bm.gpu_precompute_v1
    original_run_gpu_precompute = versions_mod._run_gpu_precompute
    versions_mod._NOTEBOOK_PRECOMPUTE_MODE = None

    @contextlib.contextmanager
    def _force_precompute_mode(mode: str):
        previous = getattr(versions_mod, '_NOTEBOOK_PRECOMPUTE_MODE', None)
        versions_mod._NOTEBOOK_PRECOMPUTE_MODE = str(mode).strip().lower()
        try:
            yield
        finally:
            versions_mod._NOTEBOOK_PRECOMPUTE_MODE = previous

    def _run_gpu_precompute_with_policy(
        backend, solver, gpu_cfg, data_ctx, op_ctx, *, use_original_precompute
    ):
        mode = getattr(versions_mod, '_NOTEBOOK_PRECOMPUTE_MODE', None)
        if mode is None:
            return original_run_gpu_precompute(
                backend,
                solver,
                gpu_cfg,
                data_ctx,
                op_ctx,
                use_original_precompute=use_original_precompute,
            )

        previous_active = getattr(bench_mod, '_BENCHMARK_PC_METHOD_ACTIVE', None)
        try:
            bench_mod._BENCHMARK_PC_METHOD_ACTIVE = 'original' if mode == 'original' else 'c1'
            return patched_gpu_precompute(
                backend,
                solver.kernel,
                solver.eps,
                solver.nufft_tol,
                data_ctx,
                op_ctx,
                l2scaled=solver.l2scaled,
                chunk_size=gpu_cfg.chunk_size,
            )
        finally:
            bench_mod._BENCHMARK_PC_METHOD_ACTIVE = previous_active

    def _wrap_runner(fn, mode: str):
        def _wrapped(*args, **kwargs):
            with _force_precompute_mode(mode):
                return fn(*args, **kwargs)
        _wrapped.__name__ = getattr(fn, '__name__', 'wrapped_runner')
        _wrapped.__doc__ = getattr(fn, '__doc__', None)
        return _wrapped

    versions_mod._run_gpu_precompute = _run_gpu_precompute_with_policy
    run_mod.run_v1_pure_efgp = _wrap_runner(
        run_mod.run_v1_pure_efgp, PRECOMPUTE_POLICY['plain_cg']
    )
    run_mod.run_v3_full_gpu_eigenspace = _wrap_runner(
        run_mod.run_v3_full_gpu_eigenspace, PRECOMPUTE_POLICY['eigenpro_pcg']
    )
    run_mod.run_v6_box_toeplitz_active_block = _wrap_runner(
        run_mod.run_v6_box_toeplitz_active_block, PRECOMPUTE_POLICY['btab_inverse']
    )
    run_mod.run_v7_box_eigenpro_active_block = _wrap_runner(
        run_mod.run_v7_box_eigenpro_active_block, PRECOMPUTE_POLICY['btab_boxeig']
    )
    return dict(PRECOMPUTE_POLICY)

print('Notebook precompute policy prepared:', PRECOMPUTE_POLICY)


In [ ]:
LEGACY_OUTPUT_ROOT = DRIVE_RUN_ROOT / "legacy_archived_pipeline"
LEGACY_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
legacy_group_outputs = []

legacy_sizes = {
    "group_a": {1_000_000, 3_000_000, 10_000_000, 30_000_000},
    "group_b": {1_000_000, 3_000_000, 10_000_000, 30_000_000, 100_000_000, 300_000_000},
    "group_c": {100_000_000, 300_000_000},
}
requested_legacy_sizes = set().union(*(
    legacy_sizes.get(group, set()) for group in RUN_LEGACY_GROUPS
)) if RUN_LEGACY_GROUPS else set()
if requested_legacy_sizes:
    validate_archived_synthetic_inputs(requested_legacy_sizes)

for group_name in RUN_LEGACY_GROUPS:
    if group_name not in {"group_a", "group_b", "group_c"}:
        raise ValueError(f"Unknown legacy group: {group_name}")
    group_out = LEGACY_OUTPUT_ROOT / group_name
    success_marker = group_out / "_SUCCESS.json"
    if success_marker.is_file():
        marker = json.loads(success_marker.read_text(encoding="utf-8"))
        current_data_manifest_sha = hashlib.sha256(DATA_MANIFEST.read_bytes()).hexdigest()
        if marker.get("git_sha") != GIT_SHA or marker.get("data_manifest_sha256") != current_data_manifest_sha:
            raise RuntimeError(
                f"{success_marker} belongs to different code/data; change RUN_TAG instead of mixing runs."
            )
        print("Legacy group already complete; skipping:", group_name)
        legacy_group_outputs.append(group_out)
        continue

    mods = reload_legacy_modules()
    install_notebook_precompute_policy(mods)
    run_experiments = mods[f"{PACKAGE}.run_experiments"].run_experiments
    cfg = BTABExperimentConfig(
        btab_experiment_route=group_name,
        btab_experiment_routes=[],
        output_dir=str(group_out),
        run_tag=f"colab_{group_name}",
        tol=1e-7,
        maxiter=80_000,
        non_v1_maxiter=3_000,
        backend=BackendConfig(xp="cupy", fft="cupy", nufft="cufinufft", linalg="cupy"),
    )
    result = run_experiments(cfg)
    completed_output = Path(result["output_dir"])
    required_outputs = [
        completed_output / "master_summary.csv",
        completed_output / "aggregate_summary.csv",
        completed_output / "experiment_config.json",
    ]
    missing_outputs = [str(path) for path in required_outputs if not path.is_file()]
    if missing_outputs:
        raise RuntimeError(f"Legacy group finished without required outputs: {missing_outputs}")
    marker_payload = {
        "protocol_family": "archived_complete_pipeline",
        "group": group_name,
        "git_sha": GIT_SHA,
        "data_manifest_sha256": hashlib.sha256(DATA_MANIFEST.read_bytes()).hexdigest(),
        "completed_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "required_outputs": [path.name for path in required_outputs],
    }
    marker_tmp = success_marker.with_suffix(".json.partial")
    marker_tmp.write_text(json.dumps(marker_payload, indent=2), encoding="utf-8")
    marker_tmp.replace(success_marker)
    legacy_group_outputs.append(completed_output)
print("Legacy outputs:", legacy_group_outputs)


In [ ]:
legacy_frames = []
for output in legacy_group_outputs:
    for candidate in [output / "master_summary.csv", output / "aggregate_summary.csv"]:
        if candidate.is_file():
            frame = pd.read_csv(candidate)
            frame["protocol_family"] = "archived_complete_pipeline"
            frame["evidence_role"] = "exploratory_scale_map"
            frame["timing_scope"] = "complete pipeline; direct CG and binned candidates use different setup routes"
            frame["legacy_group"] = output.name
            legacy_frames.append(frame)
            break
legacy_all = pd.concat(legacy_frames, ignore_index=True, sort=False) if legacy_frames else pd.DataFrame()
if not legacy_all.empty:
    legacy_all.to_csv(LEGACY_OUTPUT_ROOT / "legacy_all_groups.csv", index=False)
    display(legacy_all.head(50))
else:
    print("No legacy groups selected or completed.")


# B. 新增 controlled fixed-system 实验

每个 case 在单一 invocation 中构造一个不可变系统，所有方法共享系统哈希；每次从零初值开始，1 次预热后做 5 次随机顺序配对。Scale master 使用 `subset_mode='prefix'`：10M/30M/100M/300M 是同一 300M master 的严格行前缀。

- `screen_10m`：CG-only difficulty gate，与正式表分开。
- `paper_10m`：archived Synthetic、Winnebago、Manitowoc 三个 10M q256 center。
- `winnebago_box_budget_n10m`：同一 10M Winnebago 系统上的 4096/8192/16384 memory-budget 消融。
- `scale_archived_exact`：一键模式只选择 Winnebago 原数据定义的 exact artifacts；不同 N 不宣称嵌套。
- `scale_development_masters`：一键模式只选择 low-noise Synthetic 的同一 300M master 前缀；已知失败的 Winnebago raw-prefix 不进入正式规模任务。
- `scale_manitowoc_master`：需先准备 Manitowoc 300M master。
- `winnebago_oat_n10m`：只改变一个 λ 或 ℓ，并配对比较 CG/default/full-eig。


In [ ]:
import copy
CONTROLLED_DIR = LOCAL_REPO / "efgp_eigenpro_py/gpu/box_toeplitz_active_block/controlled"
SUITE_TEMPLATE = CONTROLLED_DIR / "colab_all_experiments_suite.json"
CONTROLLED_OUTPUT_ROOT = DRIVE_RUN_ROOT / "controlled_fixed_system"
RUNTIME_CONFIG_ROOT = DRIVE_RUN_ROOT / "runtime_configs"
CONTROLLED_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
expected_controlled_case_count = 0
selected_case_records = []
base_suite = json.loads(SUITE_TEMPLATE.read_text(encoding="utf-8"))
controlled_profile_outputs = []

def select_profile_cases(profile, *, sizes, families=(), case_ids=()):
    size_set = {int(n) for n in sizes}
    family_set = {str(name) for name in families}
    id_set = {str(case_id) for case_id in case_ids}
    selected = []
    for case in profile["cases"]:
        if int(case["expected_n_train"]) not in size_set:
            continue
        if family_set and str(case.get("dataset_family", "")) not in family_set:
            continue
        if id_set and str(case["id"]) not in id_set:
            continue
        selected.append(case)
    return selected

formal_jobs = []
if RUN_ALL_FORMAL_EXPERIMENTS:
    formal_jobs.extend([
        {"job_id": "paper_10m", "profile": "paper_10m", "sizes": [10_000_000], "mandatory": True},
        {"job_id": "winnebago_oat_n10m", "profile": "winnebago_oat_n10m", "sizes": [10_000_000], "mandatory": True},
        {"job_id": "winnebago_box_budget_n10m", "profile": "winnebago_box_budget_n10m", "sizes": [10_000_000], "mandatory": True},
    ])
    for family, profile_name, series in [
        ("Synthetic", "scale_development_masters", "synthetic_nested"),
        ("Winnebago", "scale_archived_exact", "winnebago_exact"),
    ]:
        for n_train in FORMAL_SCALE_SIZES:
            formal_jobs.append({
                "job_id": f"{series}_{n_train // 1_000_000}m",
                "profile": profile_name,
                "sizes": [int(n_train)],
                "families": [family],
                "mandatory": True,
                "gate_series": series,
                "requires_gate_n": (
                    30_000_000 if n_train == 100_000_000
                    else 100_000_000 if n_train == 300_000_000
                    else None
                ),
            })
else:
    manual_profiles = []
    if RUN_CG_SCREEN_10M: manual_profiles.append("screen_10m")
    if RUN_Q256_CENTER_10M: manual_profiles.append("paper_10m")
    if RUN_BOX_BUDGET_ABLATION: manual_profiles.append("winnebago_box_budget_n10m")
    if RUN_WINNEBAGO_OAT_10M: manual_profiles.append("winnebago_oat_n10m")
    if RUN_DEVELOPMENT_MASTER_SCALE: manual_profiles.append("scale_development_masters")
    if RUN_ARCHIVED_EXACT_SCALE: manual_profiles.append("scale_archived_exact")
    if RUN_MANITOWOC_SCALE: manual_profiles.append("scale_manitowoc_master")
    formal_jobs.extend({
        "job_id": profile_name,
        "profile": profile_name,
        "sizes": list(ACTIVE_SIZES),
        "families": list(PROFILE_DATASET_FAMILIES.get(profile_name, [])),
        "case_ids": list(ACTIVE_CASE_IDS),
        "mandatory": True,
    } for profile_name in manual_profiles)

selected_profiles = list(dict.fromkeys(job["profile"] for job in formal_jobs))
CAMPAIGN_EXECUTION_STARTED_UTC = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
CAMPAIGN_EXECUTION_STARTED_PERF_COUNTER = time.perf_counter()
CAMPAIGN_JOBS_CSV = DRIVE_RUN_ROOT / "campaign_jobs.csv"
CAMPAIGN_JOBS_JSON = DRIVE_RUN_ROOT / "campaign_jobs.json"
previous_campaign_rows_by_job = {}
if CAMPAIGN_JOBS_JSON.is_file():
    try:
        previous_rows = json.loads(CAMPAIGN_JOBS_JSON.read_text(encoding="utf-8"))
        if not isinstance(previous_rows, list) or not all(
            isinstance(row, dict) for row in previous_rows
        ):
            raise TypeError("campaign ledger is not a list of objects")
        previous_campaign_rows_by_job = {
            str(row["job_id"]): row for row in previous_rows if row.get("job_id")
        }
    except (OSError, json.JSONDecodeError, TypeError):
        print("WARNING: previous campaign ledger is unreadable; first-run timing cannot be carried forward.")
campaign_job_rows = []
planned_campaign_job_ids = (
    (["plumbing_smoke"] if RUN_PLUMBING_SMOKE else [])
    + [str(job["job_id"]) for job in formal_jobs]
    + (
        ["prediction_accuracy_audit"]
        if globals().get("RUN_PREDICTION_AUDIT", False) else []
    )
)
if RUN_PLUMBING_SMOKE:
    smoke_elapsed_seconds = globals().get("SMOKE_ELAPSED_SECONDS")
    campaign_job_rows.append({
        "job_id": "plumbing_smoke",
        "profile": "benchmark_smoke",
        "dataset_family": "Manitowoc",
        "n_train": 5000,
        "mandatory": True,
        "return_code": SMOKE_RETURN_CODE,
        "status": "PASS" if SMOKE_OK else "EXECUTION_ERROR",
        "reason": "" if SMOKE_OK else "smoke command returned non-zero",
        "case_count": 1,
        "elapsed_seconds": smoke_elapsed_seconds,
        "current_invocation_elapsed_seconds": smoke_elapsed_seconds,
        "elapsed_seconds_scope": "current smoke invocation wall time; never method timing",
        "invocation_mode": "executed",
        "resumed_case_count": 0,
        "executed_case_count": 1,
        "first_run_elapsed_seconds": (
            smoke_elapsed_seconds if SMOKE_OK else None
        ),
        "first_run_elapsed_seconds_source": (
            "current successful invocation"
            if SMOKE_OK else "unavailable: smoke failed"
        ),
    })

def write_campaign_checkpoint():
    # Preserve first-run provenance for planned jobs that this
    # resumed invocation has not replayed yet. Otherwise a second
    # interruption after the first job would truncate the old
    # ledger to the current prefix.
    current_by_job = {
        str(row["job_id"]): row
        for row in campaign_job_rows if row.get("job_id")
    }
    checkpoint_rows = [
        current_by_job.get(job_id)
        or previous_campaign_rows_by_job.get(job_id)
        for job_id in planned_campaign_job_ids
        if (
            job_id in current_by_job
            or job_id in previous_campaign_rows_by_job
        )
    ]
    checkpoint_rows.extend(
        row for row in campaign_job_rows
        if str(row.get("job_id")) not in set(planned_campaign_job_ids)
    )
    payload = json.dumps(checkpoint_rows, indent=2)
    partial_json = CAMPAIGN_JOBS_JSON.with_suffix(".json.partial")
    partial_json.write_text(payload, encoding="utf-8")
    partial_json.replace(CAMPAIGN_JOBS_JSON)
    frame = pd.DataFrame(checkpoint_rows)
    partial_csv = CAMPAIGN_JOBS_CSV.with_suffix(".csv.partial")
    frame.to_csv(partial_csv, index=False)
    partial_csv.replace(CAMPAIGN_JOBS_CSV)

def classify_suite_invocation(return_code, status_rows, expected_case_count):
    terminal = [str(row.get("status", "")) for row in status_rows]
    complete_status = (
        len(status_rows) == int(expected_case_count)
        and all(status not in {"", "running"} for status in terminal)
    )
    hard_error = any(status == "error" for status in terminal)
    scientific_failure = any(
        row.get("ineligible_methods") or row.get("diagnostic_errors")
        for row in status_rows
    )
    if not complete_status or hard_error:
        return "EXECUTION_ERROR", "missing/nonterminal case status or case-level error"
    if scientific_failure or int(return_code) == 2:
        return "SCIENTIFIC_FAIL", "complete artifacts contain ineligible methods or diagnostic errors"
    if int(return_code) == 0:
        return "PASS", ""
    # Backward compatibility with older suite.py, which returned 1 for a
    # complete scientific failure.
    if int(return_code) == 1 and scientific_failure:
        return "SCIENTIFIC_FAIL", "legacy scientific-failure exit code"
    return "EXECUTION_ERROR", f"unexpected suite return code {return_code}"

def execution_metadata_from_counts(
    *, case_count, resumed_count, elapsed_seconds,
    invocation_successful, previous_row=None,
):
    case_count = int(case_count)
    resumed_count = int(resumed_count)
    executed_count = max(case_count - resumed_count, 0)
    if case_count <= 0:
        invocation_mode = "no_cases"
    elif resumed_count == case_count:
        invocation_mode = "resumed_existing"
    elif resumed_count:
        invocation_mode = "mixed_execute_and_resume"
    else:
        invocation_mode = "executed"
    previous_row = previous_row if isinstance(previous_row, dict) else {}
    previous_first_run = (
        previous_row.get("first_run_elapsed_seconds")
        if previous_row.get("status") == "PASS" else None
    )
    if invocation_mode == "executed" and invocation_successful:
        first_run_elapsed = float(elapsed_seconds)
        first_run_source = "current successful invocation"
    elif previous_first_run is not None:
        first_run_elapsed = float(previous_first_run)
        first_run_source = "preserved successful campaign checkpoint"
    else:
        first_run_elapsed = None
        first_run_source = (
            "unavailable: current/previous fresh invocation was not successful"
        )
    return {
        "elapsed_seconds": float(elapsed_seconds),
        "current_invocation_elapsed_seconds": float(elapsed_seconds),
        "elapsed_seconds_scope": (
            "current suite invocation wall time; resume validation is not first-run "
            "experiment time and neither value is method timing"
        ),
        "invocation_mode": invocation_mode,
        "resumed_case_count": resumed_count,
        "executed_case_count": executed_count,
        "first_run_elapsed_seconds": first_run_elapsed,
        "first_run_elapsed_seconds_source": first_run_source,
    }

def suite_execution_metadata(
    status_rows, elapsed_seconds, job_status, previous_row=None,
):
    statuses = [str(row.get("status", "")) for row in status_rows]
    metadata = execution_metadata_from_counts(
        case_count=len(statuses),
        resumed_count=sum(
            status.startswith("resumed_existing") for status in statuses
        ),
        elapsed_seconds=elapsed_seconds,
        invocation_successful=(job_status == "PASS"),
        previous_row=previous_row,
    )
    metadata["suite_case_statuses"] = statuses
    return metadata

def scale_core_pass(case_records):
    required = {"cg", "default", "full-eig"}
    for record in case_records:
        summary_path = Path(record["run_dir"]) / "matched_summary.csv"
        config_path = Path(record["run_dir"]) / "experiment_config.json"
        if not summary_path.is_file() or not config_path.is_file():
            return False
        summary = pd.read_csv(summary_path)
        config = json.loads(config_path.read_text(encoding="utf-8"))
        core = summary.loc[summary["method"].astype(str).isin(required)].copy()
        if set(core["method"].astype(str)) != required:
            return False
        expected_repeats = int(config.get("measured_repeats", 5))
        eligible = core["performance_claim_eligible"].astype(str).str.lower().eq("true")
        converged = pd.to_numeric(core["converged_repeats"], errors="coerce").eq(expected_repeats)
        residual_ok = pd.to_numeric(core["true_relres_max"], errors="coerce").le(
            float(config.get("tol", 1e-7))
        )
        if not bool((eligible & converged & residual_ok).all()):
            return False
    return True

gate_results = {}
for job in formal_jobs:
    job_id = str(job["job_id"])
    profile_name = str(job["profile"])
    sizes = [int(n) for n in job["sizes"]]
    families = list(job.get("families", PROFILE_DATASET_FAMILIES.get(profile_name, [])))
    started = time.perf_counter()
    base_row = {
        "job_id": job_id,
        "profile": profile_name,
        "dataset_family": ",".join(families),
        "n_train": ",".join(str(n) for n in sizes),
        "mandatory": bool(job.get("mandatory", True)),
    }

    skip_reason = ""
    if not SMOKE_OK:
        skip_reason = "SKIPPED_SMOKE_FAILED"
    elif 300_000_000 in sizes and not CAN_RUN_300M:
        skip_reason = "SKIPPED_HARDWARE"
    required_gate_n = job.get("requires_gate_n")
    if required_gate_n is not None:
        gate_key = (str(job.get("gate_series")), int(required_gate_n))
        if gate_results.get(gate_key) is not True:
            skip_reason = "SKIPPED_UPSTREAM_GATE"
    if skip_reason:
        elapsed = time.perf_counter() - started
        campaign_job_rows.append({
            **base_row, "return_code": None, "status": skip_reason,
            "reason": skip_reason, "case_count": 0,
            "elapsed_seconds": elapsed,
            "current_invocation_elapsed_seconds": elapsed,
            "elapsed_seconds_scope": "gate evaluation only; no experiment executed",
            "invocation_mode": "skipped",
            "resumed_case_count": 0,
            "executed_case_count": 0,
            "first_run_elapsed_seconds": None,
            "first_run_elapsed_seconds_source": "not executed",
        })
        write_campaign_checkpoint()
        print(f"[{job_id}] {skip_reason}")
        continue

    profile = copy.deepcopy(base_suite["profiles"][profile_name])
    cases = select_profile_cases(
        profile,
        sizes=sizes,
        families=families,
        case_ids=job.get("case_ids", ()),
    )
    if not cases:
        elapsed = time.perf_counter() - started
        campaign_job_rows.append({
            **base_row, "return_code": None, "status": "CONFIG_ERROR",
            "reason": "profile filter selected zero cases", "case_count": 0,
            "elapsed_seconds": elapsed,
            "current_invocation_elapsed_seconds": elapsed,
            "elapsed_seconds_scope": "configuration validation only; no experiment executed",
            "invocation_mode": "no_cases",
            "resumed_case_count": 0,
            "executed_case_count": 0,
            "first_run_elapsed_seconds": None,
            "first_run_elapsed_seconds_source": "not executed",
        })
        write_campaign_checkpoint()
        print(f"[{job_id}] CONFIG_ERROR: profile filter selected zero cases")
        if job.get("gate_series"):
            gate_results[(str(job["gate_series"]), sizes[0])] = False
        continue

    if profile_name == "scale_archived_exact":
        synthetic_sizes = [
            case["expected_n_train"] for case in cases
            if case.get("dataset_family") == "Synthetic"
        ]
        if synthetic_sizes:
            validate_archived_synthetic_inputs(synthetic_sizes)
    profile["cases"] = cases
    runtime_suite = {
        "base": copy.deepcopy(base_suite["base"]),
        "profiles": {profile_name: profile},
    }
    runtime_path = RUNTIME_CONFIG_ROOT / f"{job_id}.json"
    runtime_path.write_text(json.dumps(runtime_suite, indent=2), encoding="utf-8")
    output_root = CONTROLLED_OUTPUT_ROOT / profile_name / "_jobs" / job_id
    output_root.mkdir(parents=True, exist_ok=True)
    job_case_records = []
    for case in cases:
        record = {
            "output_group": profile_name,
            "suite_profile": profile_name,
            "job_id": job_id,
            "case_id": case["id"],
            "dataset_family": case.get("dataset_family", ""),
            "run_dir": output_root / case["id"],
            "scale_role": case.get("scale_role"),
            "mandatory": bool(job.get("mandatory", True)),
        }
        selected_case_records.append(record)
        job_case_records.append(record)
    expected_controlled_case_count += len(cases)

    completed = run_cmd([
        sys.executable, "-m",
        "efgp_eigenpro_py.gpu.box_toeplitz_active_block.controlled.suite",
        "--config", str(runtime_path), "--profile", profile_name,
        "--dataset-dir", str(LOCAL_DATA_DIR),
        "--output-root", str(output_root),
        "--nufft-backend", "cufinufft", "--strict-gpu-eig",
        "--execute", "--resume",
    ], cwd=LOCAL_REPO, check=False)
    status_path = output_root / "suite_status.json"
    try:
        status_rows = json.loads(status_path.read_text(encoding="utf-8"))
        if not isinstance(status_rows, list) or not all(
            isinstance(row, dict) for row in status_rows
        ):
            raise TypeError("suite status is not a list of objects")
    except (OSError, json.JSONDecodeError, TypeError):
        status_rows = []
    job_status, reason = classify_suite_invocation(
        completed.returncode, status_rows, len(cases)
    )
    elapsed = time.perf_counter() - started
    execution_metadata = suite_execution_metadata(
        status_rows,
        elapsed,
        job_status,
        previous_campaign_rows_by_job.get(job_id),
    )
    campaign_job_rows.append({
        **base_row,
        "return_code": int(completed.returncode),
        "status": job_status,
        "reason": reason,
        "case_count": len(cases),
        "suite_status_path": str(status_path),
        **execution_metadata,
    })
    controlled_profile_outputs.append(output_root)
    if job.get("gate_series"):
        gate_results[(str(job["gate_series"]), sizes[0])] = scale_core_pass(job_case_records)
    write_campaign_checkpoint()
    print(f"[{job_id}] {job_status}: {reason or 'all selected cases passed'}")

print("Controlled job outputs:", [str(path) for path in controlled_profile_outputs])
display(pd.DataFrame(campaign_job_rows))


## B.1 q128 bridge 与 SE full-inverse control（可选）

两个历史 suite 的 Synthetic stem 默认指向 low-noise `_n10000000`。本格在运行时改成 archived `_ntrain10000000`，不修改原模板。两种 control 仍各自保持同一 case 内固定 (A,b)。


In [ ]:
extra_suites = []
if RUN_Q128_BRIDGE:
    extra_suites.append(("q128_bridge", "original_data_n10m_matched_bridge_suite.json", "bridge"))
if RUN_SE_FULL_INVERSE_CONTROL:
    extra_suites.append(("se_full_inverse", "original_data_n10m_se_full_inverse_suite.json", "se_control"))

for label, filename, profile_name in extra_suites:
    payload = json.loads((CONTROLLED_DIR / filename).read_text(encoding="utf-8"))
    for profile in payload["profiles"].values():
        for case in profile["cases"]:
            if case.get("dataset_stem") == "synthetic_true_func_2d_n10000000":
                case["dataset_stem"] = "synthetic_true_func_2d_ntrain10000000"
    runtime_path = RUNTIME_CONFIG_ROOT / f"{label}.json"
    runtime_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    output_root = CONTROLLED_OUTPUT_ROOT / label
    extra_cases = payload["profiles"][profile_name]["cases"]
    expected_controlled_case_count += len(extra_cases)
    for case in extra_cases:
        selected_case_records.append({
            "output_group": label,
            "suite_profile": profile_name,
            "case_id": case["id"],
            "run_dir": output_root / case["id"],
            "scale_role": case.get("scale_role"),
        })
    run_cmd([
        sys.executable, "-m",
        "efgp_eigenpro_py.gpu.box_toeplitz_active_block.controlled.suite",
        "--config", str(runtime_path), "--profile", profile_name,
        "--dataset-dir", str(LOCAL_DATA_DIR), "--output-root", str(output_root),
        "--nufft-backend", "cufinufft", "--strict-gpu-eig",
        "--execute", "--resume",
    ], cwd=LOCAL_REPO)
    controlled_profile_outputs.append(output_root)


## B.2 正式 controlled artifact 审计

cuFINUFFT adapter 可能在运行期失败后回退 CPU，因此不能只检查请求参数；必须同时检查 manifest 的 `nufft_backend_resolved` 和 `nufft_stage`。下格也检查 fp64、system unchanged、strict GPU eig 和完整 `run_complete.json`。


In [ ]:
expected_controlled_case_count = len(selected_case_records)
selected_run_dirs = {
    Path(record["run_dir"]).resolve() for record in selected_case_records
}
discovered_run_dirs = {
    path.parent.resolve()
    for path in CONTROLLED_OUTPUT_ROOT.rglob("system_manifest.json")
}
ignored_stale_dirs = sorted(discovered_run_dirs - selected_run_dirs, key=str)
audit_rows = []
for record in selected_case_records:
    run_dir = Path(record["run_dir"]).resolve()
    manifest_path = run_dir / "system_manifest.json"
    config_path = run_dir / "experiment_config.json"
    complete_path = run_dir / "run_complete.json"
    summary_path = run_dir / "matched_summary.csv"
    problems = []
    warnings = []
    manifest = {}
    config = {}
    if not manifest_path.is_file():
        problems.append("missing system_manifest.json")
    else:
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            manifest = {}
            problems.append("unreadable system_manifest.json")
        if not isinstance(manifest, dict):
            manifest = {}
            problems.append("system_manifest.json is not an object")
        if not manifest.get("system_unchanged"): problems.append("system changed")
        if manifest.get("nufft_backend_resolved") != "cufinufft": problems.append("backend fallback")
        if manifest.get("nufft_stage") != "cufinufft": problems.append("NUFFT stage is not GPU")
        if manifest.get("precision_mode") != "fp64": problems.append("not fp64")
        if not manifest.get("device_name"): problems.append("missing timing device_name")
        if not manifest.get("compute_capability"): problems.append("missing compute_capability")
        if not manifest.get("timing_runtime_sha256"): problems.append("missing timing_runtime_sha256")
        missing_component_hashes = [
            field for field in (
                "weights_sha256", "gf_sha256", "rhs_sha256",
                "rhs_storage_sha256",
            )
            if not manifest.get(field)
        ]
        if missing_component_hashes:
            problems.append(
                "manifest lacks exact component hashes: "
                + ", ".join(missing_component_hashes)
            )
    if not config_path.is_file():
        problems.append("missing experiment_config.json")
    else:
        try:
            config = json.loads(config_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            config = {}
            problems.append("unreadable experiment_config.json")
        if not isinstance(config, dict):
            config = {}
            problems.append("experiment_config.json is not an object")
        if not config.get("strict_gpu_eig"): problems.append("strict_gpu_eig false")
    if not complete_path.is_file(): problems.append("missing run_complete.json")
    if not summary_path.is_file():
        problems.append("missing matched_summary.csv")
    else:
        try:
            matched = pd.read_csv(summary_path)
        except (OSError, pd.errors.ParserError, pd.errors.EmptyDataError) as exc:
            matched = pd.DataFrame()
            problems.append(f"unreadable matched_summary.csv: {type(exc).__name__}")
        required_columns = {
            "method", "measured_repeats", "converged_repeats",
            "performance_claim_eligible", "true_relres_max", "cold_speedup_median",
        }
        if matched.empty:
            problems.append("matched summary is empty")
        elif not required_columns.issubset(matched.columns):
            problems.append("matched summary lacks eligibility columns")
        else:
            expected_methods = config.get("methods")
            observed_methods = matched["method"].astype(str).tolist()
            if (
                not isinstance(expected_methods, list)
                or not expected_methods
                or len({str(method) for method in expected_methods})
                != len(expected_methods)
                or sorted(observed_methods)
                != sorted(str(method) for method in expected_methods)
            ):
                problems.append(
                    "matched summary method coverage differs from experiment_config.methods"
                )
            eligible = matched["performance_claim_eligible"].astype(str).str.lower().eq("true")
            expected_repeats = int(config.get("measured_repeats", 5))
            five_of_five = (
                matched["measured_repeats"].astype(int).eq(expected_repeats)
                & matched["converged_repeats"].astype(int).eq(expected_repeats)
            )
            true_relres_ok = pd.to_numeric(
                matched["true_relres_max"], errors="coerce"
            ).le(float(config.get("tol", 1e-7)))
            claim_ok = eligible & five_of_five & true_relres_ok
            if not bool(claim_ok.all()):
                bad_methods = matched.loc[~claim_ok, "method"].astype(str).tolist()
                problems.append(f"ineligible/nonconverged methods: {bad_methods}")
            cg = matched.loc[matched["method"].astype(str).eq("cg")]
            cg_speed = pd.to_numeric(cg.get("cold_speedup_median"), errors="coerce")
            if len(cg) != 1 or not bool((cg_speed - 1.0).abs().le(1e-12).all()):
                problems.append("CG cold speedup is not exactly one")
            if {"build_seconds_median", "build_seconds_max"}.issubset(matched.columns):
                build_median = pd.to_numeric(matched["build_seconds_median"], errors="coerce")
                build_max = pd.to_numeric(matched["build_seconds_max"], errors="coerce")
                jitter = build_median.gt(0.01) & build_max.gt(1.5 * build_median)
                if bool(jitter.any()):
                    warnings.append(
                        "build-time spikes: "
                        + ", ".join(matched.loc[jitter, "method"].astype(str).tolist())
                    )
    audit_rows.append({
        "run_dir": str(run_dir),
        "output_group": record["output_group"],
        "suite_profile": record["suite_profile"],
        "job_id": record.get("job_id", record["suite_profile"]),
        "mandatory": bool(record.get("mandatory", True)),
        "case": record["case_id"],
        "N": manifest.get("n_train"),
        "system_id": manifest.get("system_id"),
        "weights_sha256": manifest.get("weights_sha256"),
        "gf_sha256": manifest.get("gf_sha256"),
        "rhs_sha256": manifest.get("rhs_sha256"),
        "rhs_storage_sha256": manifest.get("rhs_storage_sha256"),
        "device_name": manifest.get("device_name"),
        "compute_capability": manifest.get("compute_capability"),
        "timing_runtime_sha256": manifest.get("timing_runtime_sha256"),
        "warning": "; ".join(warnings),
        "status": "PASS" if not problems else "FAIL: " + "; ".join(problems),
    })
controlled_artifact_audit = pd.DataFrame(
    audit_rows,
    columns=[
        "run_dir", "output_group", "suite_profile", "job_id", "mandatory", "case", "N", "system_id",
        "weights_sha256", "gf_sha256", "rhs_sha256", "rhs_storage_sha256",
        "device_name", "compute_capability", "timing_runtime_sha256",
        "warning", "status",
    ],
)
if not controlled_artifact_audit.empty:
    for output_group, group_index in controlled_artifact_audit.groupby(
        "output_group", sort=False
    ).groups.items():
        group = controlled_artifact_audit.loc[group_index]
        if (
            group["device_name"].dropna().astype(str).nunique() != 1
            or group["compute_capability"].dropna().astype(str).nunique() != 1
            or group["timing_runtime_sha256"].dropna().astype(str).nunique() != 1
        ):
            controlled_artifact_audit.loc[group_index, "status"] = (
                "FAIL: selected profile mixes or lacks GPU runtime identity"
            )
CONTROLLED_AUDIT_PATH = DRIVE_RUN_ROOT / "controlled_artifact_audit.csv"
controlled_artifact_audit.to_csv(CONTROLLED_AUDIT_PATH, index=False)
ignored_controlled_artifacts = pd.DataFrame({
    "ignored_stale_run_dir": [str(path) for path in ignored_stale_dirs]
})
IGNORED_ARTIFACTS_PATH = DRIVE_RUN_ROOT / "ignored_stale_controlled_artifacts.csv"
ignored_controlled_artifacts.to_csv(IGNORED_ARTIFACTS_PATH, index=False)
display(controlled_artifact_audit)
if ignored_stale_dirs:
    print(f"Ignored {len(ignored_stale_dirs)} stale/non-selected controlled run directories.")
if len(controlled_artifact_audit) != expected_controlled_case_count:
    print(
        "CONTROLLED AUDIT COUNT MISMATCH: "
        f"Expected exactly {expected_controlled_case_count} controlled cases, "
        f"but found {len(controlled_artifact_audit)} manifests."
    )
if not controlled_artifact_audit.empty and not controlled_artifact_audit["status"].eq("PASS").all():
    print(
        "CONTROLLED ARTIFACT AUDIT CONTAINS FAILURES. "
        "Failed rows remain in the diagnostic tables and are excluded from performance plots."
    )


# C. Prediction-equivalence audit（复用 timed system/solution；单独、非计时）

Audit 必须读取 timing case 保存的 exact system 与 canonical timed \(\beta\)，不得重建系统或重解。它只按 GPU chunk 计算预测，prediction 时间明确排除在 speedup claim 外。目标包括 `paper_10m` 三个 case 的全部 timed methods，以及 Synthetic nested-prefix 的 100M/300M（`cg/default`）；每个 case 最多使用测试集的前 2.5M 行。


In [ ]:
prediction_outputs = []
prediction_validation_rows = []
expected_prediction_case_count = 0
if RUN_PREDICTION_AUDIT:
    from efgp_eigenpro_py.gpu.box_toeplitz_active_block.controlled.prediction_audit import (
        PREDICTION_AUDIT_COMPLETION_FILENAME,
        PREDICTION_AUDIT_CSV_FILENAME,
        PREDICTION_AUDIT_JSON_FILENAME,
        prediction_source_manifest,
    )
    CURRENT_PREDICTION_SOURCE_SHA256 = prediction_source_manifest()[
        "prediction_source_bundle_sha256"
    ]
    prediction_started = time.perf_counter()
    prediction_resumed_count = 0
    prediction_records = [
        record for record in selected_case_records
        if (
            record.get("suite_profile") in set(PREDICTION_AUDIT_PROFILES)
            and (
            record.get("suite_profile") == "paper_10m"
            or (
                record.get("suite_profile") == "scale_development_masters"
                and record.get("dataset_family") == "Synthetic"
                and record.get("job_id") in {"synthetic_nested_100m", "synthetic_nested_300m"}
            )
            )
        )
    ]
    expected_prediction_case_count = len(prediction_records)
    prediction_targets = []
    prediction_errors = []
    prediction_scientific_failures = []

    def validate_prediction_payload(payload, *, audit_out, config_path, timing_manifest, required_methods, expected_n_test):
        problems = []
        if not isinstance(payload, dict):
            return ["prediction audit payload is not a JSON object"]
        current_config_sha = hashlib.sha256(config_path.read_bytes()).hexdigest()
        if payload.get("config_source_sha256") != current_config_sha:
            problems.append("config_source_sha256 mismatch")
        if payload.get("dataset_content_index_sha256") != timing_manifest.get("dataset_content_index_sha256"):
            problems.append("dataset_content_index_sha256 mismatch")
        if payload.get("source_bundle_sha256") != timing_manifest.get("source_bundle_sha256"):
            problems.append("source_bundle_sha256 mismatch")
        if payload.get("prediction_source_bundle_sha256") != CURRENT_PREDICTION_SOURCE_SHA256:
            problems.append("prediction_source_bundle_sha256 mismatch")
        if payload.get("test_dataset_content_index_verified") is not True:
            problems.append("test dataset content index was not verified")
        if payload.get("test_dataset_metadata_verified") is not True:
            problems.append("test dataset metadata was not verified")
        if payload.get("strict_prediction_nufft") is not True:
            problems.append("strict_prediction_nufft must be true")
        if payload.get("observed_prediction_nufft_stages") != ["cufinufft"]:
            problems.append("prediction did not exclusively use cufinufft")
        if payload.get("audit_rebuilt_system") is not False:
            problems.append("audit_rebuilt_system must be false")
        if payload.get("timing_system_reused") is not True:
            problems.append("timing_system_reused must be true")
        if payload.get("timing_solutions_reused") is not True:
            problems.append("timing_solutions_reused must be true")
        if payload.get("timing_solution_hashes_verified") is not True:
            problems.append("timing_solution_hashes_verified must be true")
        if payload.get("timing_system_hashes_exact") is not True:
            problems.append("timing_system_hashes_exact must be true")
        if payload.get("audit_solve_count") != 0:
            problems.append("audit_solve_count must be zero")
        if payload.get("audit_solves_per_method") != 0:
            problems.append("audit_solves_per_method must be zero")
        timing_completion_path = config_path.parent / "run_complete.json"
        if not timing_completion_path.is_file():
            problems.append("timing run_complete.json is missing")
        elif payload.get("timing_run_complete_sha256") != hashlib.sha256(
            timing_completion_path.read_bytes()
        ).hexdigest():
            problems.append("timing run completion checksum mismatch")
        if payload.get("audit_pass") is not True:
            problems.append("audit_pass is not true")
        for field in (
            "system_id", "weights_sha256", "gf_sha256", "rhs_sha256",
            "rhs_storage_sha256", "reg_lambda",
        ):
            if not timing_manifest.get(field) or payload.get(field) != timing_manifest.get(field):
                problems.append(f"{field} does not exactly match timing artifact")
        try:
            observed_n_test = int(payload.get("evaluated_n_test", -1))
        except (TypeError, ValueError):
            observed_n_test = -1
        if observed_n_test != int(expected_n_test):
            problems.append(
                f"evaluated_n_test={payload.get('evaluated_n_test')} expected={expected_n_test}"
            )
        rows = payload.get("rows", [])
        if not isinstance(rows, list) or not all(isinstance(row, dict) for row in rows):
            problems.append("prediction rows are not a list of objects")
            rows = []
        observed_methods = {str(row.get("method")) for row in rows}
        if observed_methods != set(required_methods):
            problems.append(
                f"prediction methods={sorted(observed_methods)} expected={sorted(required_methods)}"
            )
        if len(rows) != len(required_methods):
            problems.append(
                "prediction JSON must contain exactly one row per required method"
            )
        non_equivalent = sorted(
            str(row.get("method")) for row in rows
            if str(row.get("method")) != "cg"
            and row.get("prediction_equivalent_to_cg") is not True
        )
        if non_equivalent:
            problems.append(f"prediction equivalence failed: {non_equivalent}")

        json_path = Path(audit_out) / PREDICTION_AUDIT_JSON_FILENAME
        csv_path = Path(audit_out) / PREDICTION_AUDIT_CSV_FILENAME
        completion_path = Path(audit_out) / PREDICTION_AUDIT_COMPLETION_FILENAME
        if not (json_path.is_file() and csv_path.is_file() and completion_path.is_file()):
            problems.append("prediction JSON/CSV/completion artifact set is incomplete")
        else:
            try:
                completion = json.loads(completion_path.read_text(encoding="utf-8"))
                csv_frame = pd.read_csv(csv_path)
            except (OSError, json.JSONDecodeError, ValueError) as exc:
                problems.append(f"prediction completion/CSV unreadable: {type(exc).__name__}")
            else:
                if not isinstance(completion, dict):
                    problems.append("prediction completion is not a JSON object")
                else:
                    if completion.get("prediction_audit_json_sha256") != hashlib.sha256(json_path.read_bytes()).hexdigest():
                        problems.append("prediction JSON checksum mismatch")
                    if completion.get("prediction_audit_csv_sha256") != hashlib.sha256(csv_path.read_bytes()).hexdigest():
                        problems.append("prediction CSV checksum mismatch")
                    if completion.get("system_id") != payload.get("system_id"):
                        problems.append("prediction completion system_id mismatch")
                    if completion.get("methods") != list(required_methods):
                        problems.append("prediction completion methods mismatch")
                    try:
                        completion_row_count = int(completion.get("row_count", -1))
                    except (TypeError, ValueError, OverflowError):
                        completion_row_count = -1
                    if completion_row_count != len(rows):
                        problems.append("prediction completion row_count mismatch")
                    if completion.get("audit_pass") != payload.get("audit_pass"):
                        problems.append("prediction completion audit_pass mismatch")
                    if completion.get("prediction_source_bundle_sha256") != CURRENT_PREDICTION_SOURCE_SHA256:
                        problems.append("prediction completion source mismatch")
                if len(csv_frame) != len(rows) or set(csv_frame.get("method", [])) != set(required_methods):
                    problems.append("prediction CSV rows/methods differ from JSON")
        return problems

    for record in prediction_records:
        config_path = Path(record["run_dir"]) / "experiment_config.json"
        manifest_path = Path(record["run_dir"]) / "system_manifest.json"
        if not manifest_path.is_file() or not config_path.is_file():
            message = f"prediction target lacks config/manifest: {record['run_dir']}"
            prediction_errors.append(message)
            prediction_validation_rows.append({
                "case_id": record["case_id"], "status": "EXECUTION_ERROR",
                "audit_pass": False,
                "exact_timing_system_match": False,
                "timing_solutions_reused": False,
                "timing_solution_hashes_verified": False,
                "zero_audit_solves": False,
                "problems": message,
            })
            continue
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            config = json.loads(config_path.read_text(encoding="utf-8"))
            n_train = int(manifest["n_train"])
        except (OSError, json.JSONDecodeError, KeyError, TypeError, ValueError) as exc:
            message = (
                f"prediction target has unreadable config/manifest: "
                f"{record['run_dir']} ({type(exc).__name__})"
            )
            prediction_errors.append(message)
            prediction_validation_rows.append({
                "case_id": record["case_id"], "status": "EXECUTION_ERROR",
                "audit_pass": False,
                "exact_timing_system_match": False,
                "timing_solutions_reused": False,
                "timing_solution_hashes_verified": False,
                "zero_audit_solves": False,
                "problems": message,
            })
            continue
        required_methods = (
            [str(method) for method in config.get("methods", [])]
            if record.get("suite_profile") == "paper_10m"
            else ["cg", "default"]
        )
        expected_n_test = min(int(PREDICTION_AUDIT_MAX_TEST_N), n_train // 4)
        prediction_targets.append(
            (record, config_path, manifest, required_methods, expected_n_test)
        )
    for record, config_path, manifest, required_methods, expected_n_test in prediction_targets:
        audit_out = config_path.parent / "prediction_audit"
        existing_is_valid = False
        if (audit_out / PREDICTION_AUDIT_JSON_FILENAME).is_file():
            try:
                existing = json.loads(
                    (audit_out / PREDICTION_AUDIT_JSON_FILENAME).read_text(encoding="utf-8")
                )
            except (OSError, json.JSONDecodeError) as exc:
                existing = {}
                existing_problems = [
                    f"existing prediction JSON unreadable: {type(exc).__name__}"
                ]
            else:
                existing_problems = validate_prediction_payload(
                    existing,
                    audit_out=audit_out,
                    config_path=config_path,
                    timing_manifest=manifest,
                    required_methods=required_methods,
                    expected_n_test=expected_n_test,
                )
            if not existing_problems:
                print("Valid prediction audit already exists; reusing", config_path.parent.name)
                prediction_outputs.append(audit_out)
                prediction_resumed_count += 1
                prediction_validation_rows.append({
                    "case_id": record["case_id"],
                    "suite_profile": record.get("suite_profile"),
                    "dataset_family": record.get("dataset_family"),
                    "n_train": int(manifest["n_train"]),
                    "required_methods": ",".join(required_methods),
                    "evaluated_n_test": expected_n_test,
                    "status": "PASS_RESUMED",
                    "audit_pass": True,
                    "exact_timing_system_match": True,
                    "timing_solutions_reused": True,
                    "timing_solution_hashes_verified": True,
                    "zero_audit_solves": True,
                    "problems": "",
                    "audit_path": str(audit_out / "prediction_audit.json"),
                })
                existing_is_valid = True
            else:
                print(
                    "Existing prediction audit is stale/ineligible; rerunning",
                    record["case_id"], existing_problems,
                )
        if existing_is_valid:
            continue
        command = [
            sys.executable, "-m",
            "efgp_eigenpro_py.gpu.box_toeplitz_active_block.controlled.prediction_audit",
            "--config", str(config_path),
            "--prediction-chunk-size", "100000",
            "--strict-prediction-nufft",
            "--max-test", str(expected_n_test),
            "--output-dir", str(audit_out),
        ]
        if record.get("suite_profile") != "paper_10m":
            command.extend(["--methods", ",".join(required_methods)])
        completed = run_cmd(command, cwd=LOCAL_REPO, check=False)
        final_payload = {}
        final_problems = ["prediction audit JSON missing"]
        if (audit_out / PREDICTION_AUDIT_JSON_FILENAME).is_file():
            try:
                final_payload = json.loads(
                    (audit_out / PREDICTION_AUDIT_JSON_FILENAME).read_text(encoding="utf-8")
                )
            except (OSError, json.JSONDecodeError) as exc:
                final_problems = [
                    f"prediction JSON unreadable after execution: {type(exc).__name__}"
                ]
            else:
                final_problems = validate_prediction_payload(
                    final_payload,
                    audit_out=audit_out,
                    config_path=config_path,
                    timing_manifest=manifest,
                    required_methods=required_methods,
                    expected_n_test=expected_n_test,
                )
        if completed.returncode == 0 and not final_problems:
            prediction_outputs.append(audit_out)
            validation_status = "PASS_EXECUTED"
        else:
            message = (
                f"prediction audit failed for {record['case_id']} rc={completed.returncode}: "
                + "; ".join(final_problems)
            )
            prediction_errors.append(message)
            if completed.returncode == 2 or final_payload.get("audit_pass") is False:
                prediction_scientific_failures.append(message)
            validation_status = (
                "SCIENTIFIC_FAIL" if message in prediction_scientific_failures
                else "EXECUTION_ERROR"
            )
        prediction_validation_rows.append({
            "case_id": record["case_id"],
            "suite_profile": record.get("suite_profile"),
            "dataset_family": record.get("dataset_family"),
            "n_train": int(manifest["n_train"]),
            "required_methods": ",".join(required_methods),
            "evaluated_n_test": expected_n_test,
            "status": validation_status,
            "audit_pass": final_payload.get("audit_pass") is True,
            "exact_timing_system_match": all(
                final_payload.get(field) == manifest.get(field)
                and manifest.get(field) is not None
                for field in (
                    "system_id", "weights_sha256", "gf_sha256",
                    "rhs_sha256", "rhs_storage_sha256", "reg_lambda",
                )
            ),
            "timing_solutions_reused": final_payload.get("timing_solutions_reused") is True,
            "timing_solution_hashes_verified": (
                final_payload.get("timing_solution_hashes_verified") is True
            ),
            "zero_audit_solves": (
                final_payload.get("audit_solve_count") == 0
                and final_payload.get("audit_solves_per_method") == 0
            ),
            "problems": "; ".join(final_problems),
            "audit_path": str(audit_out / "prediction_audit.json"),
        })
    prediction_status = (
        "PASS" if expected_prediction_case_count > 0
        and len(prediction_outputs) == expected_prediction_case_count
        and not prediction_errors
        else ("SCIENTIFIC_FAIL" if prediction_scientific_failures else "EXECUTION_ERROR")
    )
    prediction_elapsed = time.perf_counter() - prediction_started
    prediction_execution_metadata = execution_metadata_from_counts(
        case_count=expected_prediction_case_count,
        resumed_count=prediction_resumed_count,
        elapsed_seconds=prediction_elapsed,
        invocation_successful=(prediction_status == "PASS"),
        previous_row=previous_campaign_rows_by_job.get("prediction_accuracy_audit"),
    )
    prediction_families = sorted({
        str(record.get("dataset_family"))
        for record, *_ in prediction_targets
        if record.get("dataset_family")
    })
    prediction_sizes = sorted({
        int(manifest["n_train"])
        for _, _, manifest, _, _ in prediction_targets
    })
    campaign_job_rows.append({
        "job_id": "prediction_accuracy_audit",
        "profile": ",".join(PREDICTION_AUDIT_PROFILES),
        "dataset_family": ",".join(prediction_families),
        "n_train": ",".join(str(value) for value in prediction_sizes),
        "mandatory": True,
        "return_code": 0 if prediction_status == "PASS" else (2 if prediction_status == "SCIENTIFIC_FAIL" else 1),
        "status": prediction_status,
        "reason": "; ".join(prediction_errors),
        "case_count": expected_prediction_case_count,
        **prediction_execution_metadata,
    })
    write_campaign_checkpoint()
PREDICTION_ARTIFACT_AUDIT_PATH = DRIVE_RUN_ROOT / "prediction_artifact_audit.csv"
pd.DataFrame(prediction_validation_rows).to_csv(PREDICTION_ARTIFACT_AUDIT_PATH, index=False)
print("Prediction audit outputs:", prediction_outputs)


# D. 统一结果索引与图

统一索引只负责查找和展示，不跨 protocol 计算 speedup。每行保留 `output_group/case_id`、科学配置哈希、数据与源码哈希；历史 profile 可以共存在 catalog 中，但作图必须按 profile 隔离。Controlled 表的 `cold_speedup_median` 是 selection/build+solve、排除公共 Fourier setup；`shared_fourier_setup_plus_method_speedup_median` 才是把公共 setup 加回两边后的比值。

`paper_10m` 是固定规模的方法比较，单独画带范围的分组图和速度–内存 Pareto 图；三种 `scale_*` protocol 各自分图。旧的 `controlled_scale_speedup.png` 会被标记为 deprecated，不再作为论文证据。


In [ ]:
import re
from efgp_eigenpro_py.gpu.box_toeplitz_active_block.controlled.prediction_audit import (
    PREDICTION_AUDIT_COMPLETION_FILENAME,
    PREDICTION_AUDIT_CSV_FILENAME,
    PREDICTION_AUDIT_JSON_FILENAME,
)

def dataset_family_label(stem):
    text = str(stem or "")
    lower = text.lower()
    if "synthetic_true_func_2d" in lower:
        return "Synthetic"
    if "winnebago" in lower:
        return "Winnebago"
    if "usgs_ept_wi_2county_1_b23" in lower:
        return "Manitowoc"
    return text

CONFIG_INDEX_FIELDS = (
    "kernel_family", "lengthscale", "nu", "variance", "reg_lambda",
    "fourier_eps", "nufft_tol", "l2_scaled", "tol", "maxiter",
    "precision", "subset_mode", "subset_seed", "score_tau", "box_budget",
    "inverse_max_size", "rank", "nystrom_rank", "rpcholesky_rank",
    "eig_tol", "eig_maxiter", "measured_repeats", "warmup_repeats",
    "method_order_seed", "eig_seed", "nystrom_seed", "rpcholesky_seed",
    "precompute_chunk_size", "strict_gpu_eig",
)
SYSTEM_COMPONENT_FIELDS = (
    "weights_sha256", "gf_sha256", "rhs_sha256", "rhs_storage_sha256"
)

def dataset_series_id(stem):
    return re.sub(r"_n(?:train)?\d+$", "", str(stem or ""), flags=re.IGNORECASE)
selected_record_by_dir = {
    str(Path(record["run_dir"]).resolve()): record
    for record in selected_case_records
}
prediction_outputs_declared = "prediction_outputs" in globals()
selected_prediction_audit_dirs = {
    str(Path(path).resolve())
    for path in globals().get("prediction_outputs", [])
}
result_frames = []
controlled_frames = []
prediction_frames = []
report_ingest_warnings = []
for summary_path in sorted(CONTROLLED_OUTPUT_ROOT.rglob("matched_summary.csv")):
    run_dir = summary_path.parent.resolve()
    config_path = summary_path.with_name("experiment_config.json")
    try:
        frame = pd.read_csv(summary_path)
        manifest = json.loads(
            summary_path.with_name("system_manifest.json").read_text(
                encoding="utf-8"
            )
        )
        config = json.loads(config_path.read_text(encoding="utf-8"))
        if not isinstance(manifest, dict) or not isinstance(config, dict):
            raise ValueError("manifest/config is not a JSON object")
        required_report_columns = {
            "method", "performance_claim_eligible", "measured_repeats",
            "converged_repeats", "true_relres_max", "cold_speedup_median",
        }
        if frame.empty or not required_report_columns.issubset(frame.columns):
            raise ValueError(
                "controlled summary is empty or lacks required report columns"
            )
    except (
        OSError, json.JSONDecodeError, ValueError,
        pd.errors.ParserError, pd.errors.EmptyDataError,
    ) as exc:
        report_ingest_warnings.append({
            "artifact": str(summary_path),
            "reason": f"{type(exc).__name__}: {exc}",
        })
        continue
    relative_parts = run_dir.relative_to(CONTROLLED_OUTPUT_ROOT.resolve()).parts
    output_group = relative_parts[0] if relative_parts else "unknown"
    selected_record = selected_record_by_dir.get(str(run_dir))
    frame["protocol_family"] = "controlled_fixed_system"
    frame["evidence_role"] = "paired_scale_or_replication"
    frame["timing_scope"] = "selection/build + solve; shared Fourier setup excluded from cold columns"
    frame["output_group"] = output_group
    frame["suite_profile"] = (
        selected_record.get("suite_profile") if selected_record else output_group
    )
    frame["case"] = run_dir.name
    frame["case_id"] = run_dir.name
    frame["run_dir"] = str(run_dir)
    frame["selected_in_this_invocation"] = selected_record is not None
    frame["scale_role"] = selected_record.get("scale_role") if selected_record else None
    frame["dataset_stem"] = manifest.get("dataset_stem")
    frame["dataset_family"] = dataset_family_label(manifest.get("dataset_stem"))
    frame["dataset_series_id"] = dataset_series_id(manifest.get("dataset_stem"))
    frame["N"] = manifest.get("n_train")
    frame["system_id"] = manifest.get("system_id")
    for field in SYSTEM_COMPONENT_FIELDS:
        frame[field] = manifest.get(field)
    frame["config_sha256"] = hashlib.sha256(config_path.read_bytes()).hexdigest()
    frame["source_bundle_sha256"] = manifest.get("source_bundle_sha256")
    frame["dataset_content_index_sha256"] = manifest.get("dataset_content_index_sha256")
    frame["dataset_metadata_sha256"] = manifest.get("dataset_metadata_sha256")
    frame["nufft_backend_resolved"] = manifest.get("nufft_backend_resolved")
    frame["nufft_stage"] = manifest.get("nufft_stage")
    frame["precision_mode"] = manifest.get("precision_mode")
    frame["device_name"] = manifest.get("device_name")
    frame["compute_capability"] = manifest.get("compute_capability")
    frame["timing_runtime_sha256"] = manifest.get("timing_runtime_sha256")
    frame["prepared_system_loaded_from_artifact"] = manifest.get(
        "prepared_system_loaded_from_artifact", False
    )
    frame["setup_inclusive_timing_eligible"] = manifest.get(
        "setup_inclusive_timing_eligible",
        not bool(manifest.get("prepared_system_loaded_from_artifact", False)),
    )
    for field in CONFIG_INDEX_FIELDS:
        frame[f"cfg_{field}"] = config.get(field)
    scientific_config = {field: config.get(field) for field in CONFIG_INDEX_FIELDS}
    frame["scientific_config_id"] = hashlib.sha256(
        json.dumps(scientific_config, sort_keys=True, separators=(",", ":")).encode("utf-8")
    ).hexdigest()
    expected_repeats = int(config.get("measured_repeats", 5))
    eligible = frame["performance_claim_eligible"].astype(str).str.lower().eq("true")
    artifact_eligible = bool(
        manifest.get("system_unchanged")
        and manifest.get("nufft_backend_resolved") == "cufinufft"
        and manifest.get("nufft_stage") == "cufinufft"
        and manifest.get("precision_mode") == "fp64"
        and config.get("strict_gpu_eig")
        and summary_path.with_name("run_complete.json").is_file()
    )
    frame["artifact_eligible"] = artifact_eligible
    frame["claim_eligible"] = (
        eligible
        & artifact_eligible
        & pd.to_numeric(frame["measured_repeats"], errors="coerce").eq(expected_repeats)
        & pd.to_numeric(frame["converged_repeats"], errors="coerce").eq(expected_repeats)
        & pd.to_numeric(frame["true_relres_max"], errors="coerce").le(float(config.get("tol", 1e-7)))
    )
    controlled_frames.append(frame)
    result_frames.append(frame)

for audit_path in sorted(CONTROLLED_OUTPUT_ROOT.rglob("prediction_audit.csv")):
    run_dir = audit_path.parent.parent.resolve()
    audit_payload_path = audit_path.with_suffix(".json")
    try:
        frame = pd.read_csv(audit_path)
        manifest = json.loads(
            (run_dir / "system_manifest.json").read_text(encoding="utf-8")
        )
        audit_payload = (
            json.loads(audit_payload_path.read_text(encoding="utf-8"))
            if audit_payload_path.is_file() else {}
        )
        if not isinstance(manifest, dict) or not isinstance(audit_payload, dict):
            raise ValueError("prediction/system manifest is not a JSON object")
        if frame.empty or "method" not in frame.columns:
            raise ValueError("prediction audit is empty or lacks method column")
    except (
        OSError, json.JSONDecodeError, ValueError,
        pd.errors.ParserError, pd.errors.EmptyDataError,
    ) as exc:
        report_ingest_warnings.append({
            "artifact": str(audit_path),
            "reason": f"{type(exc).__name__}: {exc}",
        })
        continue
    completion_path = audit_path.parent / PREDICTION_AUDIT_COMPLETION_FILENAME
    completion_valid = False
    if completion_path.is_file() and audit_payload_path.is_file():
        try:
            completion_payload = json.loads(
                completion_path.read_text(encoding="utf-8")
            )
            completion_valid = bool(
                isinstance(completion_payload, dict)
                and completion_payload.get("prediction_audit_json_sha256")
                == hashlib.sha256(audit_payload_path.read_bytes()).hexdigest()
                and completion_payload.get("prediction_audit_csv_sha256")
                == hashlib.sha256(audit_path.read_bytes()).hexdigest()
            )
        except (OSError, json.JSONDecodeError):
            completion_valid = False
    relative_parts = run_dir.relative_to(CONTROLLED_OUTPUT_ROOT.resolve()).parts
    output_group = relative_parts[0] if relative_parts else "unknown"
    selected_record = selected_record_by_dir.get(str(run_dir))
    csv_system_id = (
        frame["system_id"].iloc[0]
        if "system_id" in frame.columns and not frame.empty else None
    )
    frame["audit_system_id"] = audit_payload.get("system_id", csv_system_id)
    frame["timing_system_id"] = manifest.get("system_id")
    for field in SYSTEM_COMPONENT_FIELDS:
        frame[f"audit_{field}"] = audit_payload.get(field)
        frame[f"timing_{field}"] = manifest.get(field)
    exact_system_match = all(
        audit_payload.get(field) is not None
        and audit_payload.get(field) == manifest.get(field)
        for field in ("system_id", *SYSTEM_COMPONENT_FIELDS, "reg_lambda")
    )
    frame["audit_rebuilt_system"] = audit_payload.get("audit_rebuilt_system")
    frame["timing_system_reused"] = audit_payload.get("timing_system_reused")
    frame["timing_solutions_reused"] = audit_payload.get("timing_solutions_reused")
    frame["timing_solution_hashes_verified"] = audit_payload.get(
        "timing_solution_hashes_verified"
    )
    frame["audit_solve_count"] = audit_payload.get("audit_solve_count")
    frame["audit_pass"] = audit_payload.get("audit_pass")
    frame["prediction_completion_valid"] = completion_valid
    frame["exact_timing_system_match"] = exact_system_match
    row_equivalent = frame["method"].astype(str).eq("cg")
    if "prediction_equivalent_to_cg" in frame.columns:
        row_equivalent = row_equivalent | frame[
            "prediction_equivalent_to_cg"
        ].astype(str).str.lower().eq("true")
    frame["prediction_claim_eligible"] = bool(
        exact_system_match
        and audit_payload.get("audit_rebuilt_system") is False
        and audit_payload.get("timing_system_reused") is True
        and audit_payload.get("timing_solutions_reused") is True
        and audit_payload.get("timing_solution_hashes_verified") is True
        and audit_payload.get("audit_solve_count") == 0
        and audit_payload.get("audit_solves_per_method") == 0
        and audit_payload.get("audit_pass") is True
        and completion_valid
    ) & row_equivalent
    frame["protocol_family"] = "prediction_audit"
    frame["evidence_role"] = "accuracy_only"
    frame["timing_scope"] = "timed beta reused; prediction-only audit excluded from all speed claims"
    frame["output_group"] = output_group
    frame["suite_profile"] = (
        selected_record.get("suite_profile") if selected_record else output_group
    )
    frame["case"] = run_dir.name
    frame["case_id"] = run_dir.name
    frame["run_dir"] = str(run_dir)
    frame["selected_in_this_invocation"] = bool(
        selected_record is not None
        and (
            not prediction_outputs_declared
            or str(audit_path.parent.resolve())
            in selected_prediction_audit_dirs
        )
    )
    frame["dataset_stem"] = manifest.get("dataset_stem")
    frame["dataset_family"] = dataset_family_label(manifest.get("dataset_stem"))
    frame["dataset_series_id"] = dataset_series_id(manifest.get("dataset_stem"))
    frame["N"] = manifest.get("n_train")
    if "test_rmse_ratio_vs_cg" in frame.columns:
        frame["test_rmse_difference_ppm_vs_cg"] = 1e6 * (
            pd.to_numeric(frame["test_rmse_ratio_vs_cg"], errors="coerce") - 1.0
        )
        frame["test_rmse_absolute_relative_difference_vs_cg"] = (
            pd.to_numeric(frame["test_rmse_ratio_vs_cg"], errors="coerce") - 1.0
        ).abs()
    prediction_frames.append(frame)
    result_frames.append(frame)

controlled_catalog = (
    pd.concat(controlled_frames, ignore_index=True, sort=False)
    if controlled_frames else pd.DataFrame()
)
prediction_catalog = (
    pd.concat(prediction_frames, ignore_index=True, sort=False)
    if prediction_frames else pd.DataFrame()
)
selected_prediction = (
    prediction_catalog.loc[prediction_catalog["selected_in_this_invocation"]].copy()
    if not prediction_catalog.empty else pd.DataFrame()
)
PREDICTION_ACCURACY_SUMMARY_PATH = DRIVE_RUN_ROOT / "prediction_accuracy_summary.csv"
selected_prediction.to_csv(PREDICTION_ACCURACY_SUMMARY_PATH, index=False)
if not controlled_catalog.empty:
    duplicate_key = ["output_group", "case_id", "method"]
    selected_for_duplicate_check = controlled_catalog.loc[
        controlled_catalog["selected_in_this_invocation"]
    ]
    duplicated = selected_for_duplicate_check.duplicated(duplicate_key, keep=False)
    if bool(duplicated.any()):
        raise RuntimeError(
            "Duplicate controlled summary rows:\n"
            + selected_for_duplicate_check.loc[duplicated, duplicate_key].to_string(index=False)
        )
SELECTED_CONTROLLED_INDEX_PATH = DRIVE_RUN_ROOT / "selected_controlled_index.csv"
selected_controlled = (
    controlled_catalog.loc[controlled_catalog["selected_in_this_invocation"]].copy()
    if not controlled_catalog.empty else pd.DataFrame()
)
selected_controlled.to_csv(SELECTED_CONTROLLED_INDEX_PATH, index=False)
REPORT_INGEST_WARNINGS_PATH = DRIVE_RUN_ROOT / "report_ingest_warnings.csv"
pd.DataFrame(report_ingest_warnings).to_csv(
    REPORT_INGEST_WARNINGS_PATH, index=False
)
if report_ingest_warnings:
    print(
        f"Skipped {len(report_ingest_warnings)} unreadable stale/partial "
        f"report artifacts; see {REPORT_INGEST_WARNINGS_PATH}."
    )
INELIGIBLE_INDEX_PATH = DRIVE_RUN_ROOT / "controlled_ineligible_rows.csv"
ineligible_controlled = (
    controlled_catalog.loc[
        controlled_catalog["selected_in_this_invocation"]
        & ~controlled_catalog["claim_eligible"]
    ].copy()
    if not controlled_catalog.empty else pd.DataFrame()
)
ineligible_controlled.to_csv(INELIGIBLE_INDEX_PATH, index=False)

if not legacy_all.empty:
    result_frames.append(legacy_all)
all_experiments = pd.concat(result_frames, ignore_index=True, sort=False) if result_frames else pd.DataFrame()
INDEX_PATH = DRIVE_RUN_ROOT / "all_experiments_index.csv"
all_experiments.to_csv(INDEX_PATH, index=False)
print("Unified index:", INDEX_PATH)
display(all_experiments.head(100))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

METHOD_ORDER = ["cg", "jacobi", "default", "full-eig", "nystrom", "rpcholesky"]
METHOD_COLORS = {
    "cg": "#4C78A8",
    "jacobi": "#9D9D9D",
    "default": "#E45756",
    "full-eig": "#72B7B2",
    "nystrom": "#54A24B",
    "rpcholesky": "#B279A2",
}
DATASET_ORDER = ["Manitowoc", "Winnebago", "Synthetic"]
GENERATED_PLOT_PATHS = []

def assert_cg_reference_one(frame, *, context):
    cg = frame.loc[frame["method"].astype(str).eq("cg")]
    values = pd.to_numeric(cg["cold_speedup_median"], errors="coerce").to_numpy(float)
    if values.size == 0 or not np.isfinite(values).all() or not np.allclose(
        values, 1.0, rtol=0.0, atol=1e-12
    ):
        raise RuntimeError(f"{context}: CG cold speedup must be exactly one; got {values}")

def minmax_error(frame, median_field, min_field, max_field):
    median = pd.to_numeric(frame[median_field], errors="raise").to_numpy(float)
    minimum = (
        pd.to_numeric(frame[min_field], errors="coerce").to_numpy(float)
        if min_field in frame.columns else median.copy()
    )
    maximum = (
        pd.to_numeric(frame[max_field], errors="coerce").to_numpy(float)
        if max_field in frame.columns else median.copy()
    )
    minimum = np.where(np.isfinite(minimum), minimum, median)
    maximum = np.where(np.isfinite(maximum), maximum, median)
    return median, np.vstack([
        np.maximum(median - minimum, 0.0),
        np.maximum(maximum - median, 0.0),
    ])

controlled_plot = selected_controlled.copy()
if not controlled_plot.empty:
    controlled_plot = controlled_plot.loc[controlled_plot["claim_eligible"]].copy()
    if "controlled_artifact_audit" in globals():
        passed_run_dirs = set(
            controlled_artifact_audit.loc[
                controlled_artifact_audit["status"].eq("PASS"), "run_dir"
            ].astype(str)
        )
        controlled_plot = controlled_plot.loc[
            controlled_plot["run_dir"].astype(str).isin(passed_run_dirs)
        ].copy()

# paper_10m is a method comparison at one N, not a scale curve.
paper_plot = (
    controlled_plot.loc[
        controlled_plot["output_group"].eq("paper_10m")
        & pd.to_numeric(controlled_plot["N"], errors="coerce").eq(10_000_000)
    ].copy()
    if not controlled_plot.empty else pd.DataFrame()
)
if not paper_plot.empty:
    assert_cg_reference_one(paper_plot, context="paper_10m")
    paper_key = ["dataset_family", "method"]
    if bool(paper_plot.duplicated(paper_key, keep=False).any()):
        raise RuntimeError("paper_10m has duplicate dataset/method rows; refusing to aggregate them.")
    dataset_order = [name for name in DATASET_ORDER if name in set(paper_plot["dataset_family"])]
    dataset_order += sorted(set(paper_plot["dataset_family"]) - set(dataset_order))
    x = np.arange(len(dataset_order), dtype=float)
    width = 0.13
    fig, ax = plt.subplots(figsize=(11.5, 6.2))
    for method_index, method in enumerate(METHOD_ORDER):
        rows = paper_plot.loc[paper_plot["method"].eq(method)].set_index("dataset_family")
        if rows.empty:
            continue
        values = np.asarray([
            float(rows.loc[name, "cold_speedup_median"]) if name in rows.index else np.nan
            for name in dataset_order
        ])
        mins = np.asarray([
            float(rows.loc[name, "cold_speedup_min"]) if name in rows.index else np.nan
            for name in dataset_order
        ])
        maxs = np.asarray([
            float(rows.loc[name, "cold_speedup_max"]) if name in rows.index else np.nan
            for name in dataset_order
        ])
        positions = x + (method_index - (len(METHOD_ORDER) - 1) / 2) * width
        yerr = np.vstack([
            np.nan_to_num(np.maximum(values - mins, 0.0)),
            np.nan_to_num(np.maximum(maxs - values, 0.0)),
        ])
        bars = ax.bar(
            positions, values, width=width, color=METHOD_COLORS[method], label=method,
            yerr=yerr, capsize=2, linewidth=0.4, edgecolor="white",
        )
        for bar, value in zip(bars, values):
            if np.isfinite(value):
                ax.text(
                    bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.12,
                    f"{value:.2f}", ha="center", va="bottom", fontsize=7, rotation=90,
                )
    ax.axhline(1.0, color="black", lw=1, ls="--")
    ax.set_xticks(x, dataset_order)
    ax.set_ylabel("paired cold speedup over CG")
    ax.set_title("Controlled 10M: selection/build + solve (median; whiskers=min–max)")
    ax.grid(axis="y", alpha=.25)
    ax.legend(ncol=3, frameon=False, loc="upper left")
    fig.tight_layout()
    paper_plot_path = DRIVE_RUN_ROOT / "controlled_10m_method_speedup.png"
    fig.savefig(paper_plot_path, dpi=180, bbox_inches="tight")
    GENERATED_PLOT_PATHS.append(paper_plot_path)
    plt.show()
    PAPER_10M_SUMMARY_PATH = DRIVE_RUN_ROOT / "controlled_10m_method_summary.csv"
    paper_plot.sort_values(["dataset_family", "method"]).to_csv(
        PAPER_10M_SUMMARY_PATH, index=False
    )
    display(paper_plot[[
        "dataset_family", "method", "cold_speedup_median", "cold_speedup_min",
        "cold_speedup_max", "shared_fourier_setup_plus_method_speedup_median",
        "iterations_median", "build_plus_solve_seconds_median",
        "preconditioner_storage_bytes", "claim_eligible",
    ]].sort_values(["dataset_family", "method"]))

    # The active-box method is memory-capped; show the speed-memory trade-off explicitly.
    pareto = paper_plot.loc[
        paper_plot["method"].ne("cg")
        & pd.to_numeric(paper_plot["preconditioner_storage_bytes"], errors="coerce").gt(0)
    ].copy()
    if not pareto.empty:
        fig, axes = plt.subplots(
            1, len(dataset_order), figsize=(5.0 * len(dataset_order), 4.5), squeeze=False,
            sharey=True,
        )
        for ax, dataset in zip(axes[0], dataset_order):
            subset = pareto.loc[pareto["dataset_family"].eq(dataset)]
            for _, row in subset.iterrows():
                memory_mib = float(row["preconditioner_storage_bytes"]) / 2**20
                speedup = float(row["cold_speedup_median"])
                method = str(row["method"])
                ax.scatter(memory_mib, speedup, s=60, color=METHOD_COLORS.get(method, "black"))
                ax.annotate(method, (memory_mib, speedup), xytext=(4, 4),
                            textcoords="offset points", fontsize=8)
            ax.axhline(1.0, color="black", lw=1, ls="--")
            ax.set_xscale("log")
            ax.set_title(dataset)
            ax.set_xlabel("preconditioner storage (MiB, log scale)")
            ax.grid(True, alpha=.25)
        axes[0][0].set_ylabel("paired cold speedup over CG")
        fig.suptitle("Controlled 10M speed-memory trade-off", y=1.02)
        fig.tight_layout()
        pareto_path = DRIVE_RUN_ROOT / "controlled_10m_speed_memory_pareto.png"
        fig.savefig(pareto_path, dpi=180, bbox_inches="tight")
        GENERATED_PLOT_PATHS.append(pareto_path)
        plt.show()

# Accuracy-only evidence from exact timed systems and canonical timed solutions.
prediction_plot = (
    selected_prediction.loc[selected_prediction["prediction_claim_eligible"]].copy()
    if not selected_prediction.empty else pd.DataFrame()
)
if not prediction_plot.empty:
    prediction_plot["audit_case_label"] = prediction_plot.apply(
        lambda row: f"{row['dataset_family']} {int(row['N']) / 1e6:g}M",
        axis=1,
    )
    prediction_key = ["case_id", "method"]
    if bool(prediction_plot.duplicated(prediction_key, keep=False).any()):
        raise RuntimeError("Prediction audit has duplicate case/method rows.")
    audit_cases = list(dict.fromkeys(prediction_plot["audit_case_label"].astype(str)))
    audit_methods = [
        method for method in METHOD_ORDER
        if method != "cg" and method in set(prediction_plot["method"])
    ]
    x = np.arange(len(audit_cases), dtype=float)
    width = min(0.16, 0.75 / max(len(audit_methods), 1))
    fig, ax = plt.subplots(figsize=(max(10.5, 1.8 * len(audit_cases)), 5.3))
    for method_index, method in enumerate(audit_methods):
        rows = prediction_plot.loc[
            prediction_plot["method"].eq(method)
        ].set_index("audit_case_label")
        values = np.asarray([
            float(rows.loc[label, "test_rmse_difference_ppm_vs_cg"])
            if label in rows.index else np.nan
            for label in audit_cases
        ])
        positions = x + (method_index - (len(audit_methods) - 1) / 2) * width
        ax.bar(
            positions, values, width=width,
            color=METHOD_COLORS.get(method), label=method,
        )
    ax.axhline(0.0, color="black", lw=1)
    ax.set_xticks(x, audit_cases, rotation=20, ha="right")
    ax.set_ylabel("test RMSE difference vs CG (ppm)")
    ax.set_title(
        "Prediction equivalence using canonical timed solutions "
        "(accuracy only; lower absolute difference is better)"
    )
    ax.grid(axis="y", alpha=.25)
    ax.legend(ncol=min(3, max(len(audit_methods), 1)), frameon=False)
    fig.tight_layout()
    prediction_plot_path = DRIVE_RUN_ROOT / "prediction_accuracy_vs_cg.png"
    fig.savefig(prediction_plot_path, dpi=180, bbox_inches="tight")
    GENERATED_PLOT_PATHS.append(prediction_plot_path)
    plt.show()

# One-at-a-time robustness scan: lambda varies at ell=0.1, and ell varies at lambda=0.1.
oat_plot = (
    controlled_plot.loc[controlled_plot["output_group"].eq("winnebago_oat_n10m")].copy()
    if not controlled_plot.empty else pd.DataFrame()
)
if not oat_plot.empty:
    assert_cg_reference_one(oat_plot, context="winnebago_oat_n10m")
    oat_key = ["case_id", "method"]
    if bool(oat_plot.duplicated(oat_key, keep=False).any()):
        raise RuntimeError("winnebago_oat_n10m has duplicate case/method rows.")
    OAT_SUMMARY_PATH = DRIVE_RUN_ROOT / "winnebago_oat_10m_summary.csv"
    oat_plot.sort_values(["cfg_reg_lambda", "cfg_lengthscale", "method"]).to_csv(
        OAT_SUMMARY_PATH, index=False
    )
    oat_methods = [method for method in ("cg", "default", "full-eig")
                   if method in set(oat_plot["method"])]
    sweep_specs = [
        ("cfg_reg_lambda", "regularization λ", "cfg_lengthscale", 0.1),
        ("cfg_lengthscale", "lengthscale ℓ", "cfg_reg_lambda", 0.1),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.2), squeeze=False)
    for column, (x_field, x_label, fixed_field, fixed_value) in enumerate(sweep_specs):
        sweep = oat_plot.loc[
            np.isclose(
                pd.to_numeric(oat_plot[fixed_field], errors="coerce"),
                fixed_value, rtol=0.0, atol=1e-15,
            )
        ].copy()
        ax_speed = axes[0][column]
        ax_memory = axes[1][column]
        for method in oat_methods:
            group = sweep.loc[sweep["method"].eq(method)].sort_values(x_field)
            if group.empty:
                continue
            x_values = pd.to_numeric(group[x_field], errors="raise").to_numpy(float)
            y_values = pd.to_numeric(group["cold_speedup_median"], errors="raise").to_numpy(float)
            lower = np.maximum(
                y_values - pd.to_numeric(group["cold_speedup_min"], errors="raise").to_numpy(float),
                0.0,
            )
            upper = np.maximum(
                pd.to_numeric(group["cold_speedup_max"], errors="raise").to_numpy(float) - y_values,
                0.0,
            )
            ax_speed.errorbar(
                x_values, y_values, yerr=np.vstack([lower, upper]), marker="o", capsize=3,
                color=METHOD_COLORS[method], label=method,
            )
            memory = pd.to_numeric(
                group["preconditioner_storage_bytes"], errors="coerce"
            ).to_numpy(float) / 2**20
            finite_memory = np.isfinite(memory) & (memory > 0)
            if finite_memory.any():
                ax_memory.plot(
                    x_values[finite_memory], memory[finite_memory], marker="o",
                    color=METHOD_COLORS[method], label=method,
                )
        for ax in (ax_speed, ax_memory):
            ax.set_xscale("log")
            ax.set_xlabel(x_label)
            ax.grid(True, alpha=.25)
        ax_speed.axhline(1.0, color="black", lw=1, ls="--")
        ax_speed.set_ylabel("paired cold speedup over CG")
        ax_speed.set_title(f"fixed {fixed_field.replace('cfg_', '')}={fixed_value:g}")
        ax_memory.set_yscale("log")
        ax_memory.set_ylabel("preconditioner storage (MiB, log scale)")
    axes[0][0].legend(frameon=False)
    axes[1][0].legend(frameon=False)
    fig.suptitle("Winnebago 10M one-at-a-time robustness", y=1.01)
    fig.tight_layout()
    oat_plot_path = DRIVE_RUN_ROOT / "winnebago_oat_10m_speed_memory.png"
    fig.savefig(oat_plot_path, dpi=180, bbox_inches="tight")
    GENERATED_PLOT_PATHS.append(oat_plot_path)
    plt.show()

# Fixed-system active-box memory-budget ablation.
BOX_BUDGET_SYSTEM_MATCH = None
budget_plot = (
    controlled_plot.loc[
        controlled_plot["output_group"].eq("winnebago_box_budget_n10m")
    ].copy()
    if not controlled_plot.empty else pd.DataFrame()
)
if not budget_plot.empty:
    case_systems = budget_plot[[
        "case_id", "system_id", "weights_sha256", "gf_sha256", "rhs_sha256",
        "rhs_storage_sha256", "cfg_reg_lambda", "device_name", "compute_capability",
        "timing_runtime_sha256",
    ]].drop_duplicates()
    BOX_BUDGET_SYSTEM_MATCH = bool(
        len(case_systems) == 3
        and all(case_systems[field].notna().all() and case_systems[field].nunique() == 1
                for field in (
                    "system_id", "weights_sha256", "gf_sha256", "rhs_sha256",
                    "rhs_storage_sha256", "cfg_reg_lambda", "device_name",
                    "compute_capability", "timing_runtime_sha256",
                ))
    )
    if not BOX_BUDGET_SYSTEM_MATCH:
        print(
            "BOX-BUDGET AUDIT FAILED: the three budget cases do not share "
            "identical fixed-system component hashes."
        )
    budget_key = ["case_id", "method"]
    if bool(budget_plot.duplicated(budget_key, keep=False).any()):
        print("BOX-BUDGET AUDIT FAILED: duplicate case/method rows.")
        BOX_BUDGET_SYSTEM_MATCH = False
    BOX_BUDGET_SUMMARY_PATH = DRIVE_RUN_ROOT / "winnebago_box_budget_10m_summary.csv"
    budget_plot.sort_values(["cfg_box_budget", "method"]).to_csv(
        BOX_BUDGET_SUMMARY_PATH, index=False
    )
    default_budget = budget_plot.loc[budget_plot["method"].eq("default")].sort_values(
        "cfg_box_budget"
    )
    if not default_budget.empty:
        x_values = pd.to_numeric(default_budget["cfg_box_budget"], errors="raise").to_numpy(float)
        actual_sizes = pd.to_numeric(default_budget["box_size"], errors="coerce").to_numpy(float)
        speed_values, speed_yerr = minmax_error(
            default_budget, "cold_speedup_median", "cold_speedup_min", "cold_speedup_max"
        )
        iteration_values, iteration_yerr = minmax_error(
            default_budget, "iterations_median", "iterations_min", "iterations_max"
        )
        fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
        axes[0].errorbar(
            x_values, speed_values, yerr=speed_yerr,
            marker="o", capsize=3, color=METHOD_COLORS["default"],
        )
        axes[0].axhline(1.0, color="black", lw=1, ls="--")
        axes[0].set_ylabel("paired cold speedup over CG")
        axes[1].errorbar(
            x_values, iteration_values, yerr=iteration_yerr,
            marker="o", capsize=3, color=METHOD_COLORS["default"],
        )
        axes[1].set_ylabel("median PCG iterations")
        axes[2].plot(
            x_values,
            pd.to_numeric(default_budget["preconditioner_storage_bytes"], errors="raise") / 2**20,
            marker="o", color=METHOD_COLORS["default"],
        )
        axes[2].set_ylabel("preconditioner storage (MiB)")
        for ax in axes:
            ax.set_xscale("log", base=2)
            ax.set_xlabel("nominal box budget")
            ax.grid(True, alpha=.25)
            ax.set_xticks(x_values, [f"{int(x):,}" for x in x_values])
        for x_value, actual_size in zip(x_values, actual_sizes):
            if np.isfinite(actual_size):
                axes[0].annotate(
                    f"actual {int(actual_size):,}",
                    (x_value, float(default_budget.loc[
                        pd.to_numeric(default_budget["cfg_box_budget"], errors="coerce").eq(x_value),
                        "cold_speedup_median",
                    ].iloc[0])),
                    xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8,
                )
        budget_audit_label = (
            "fixed-system" if BOX_BUDGET_SYSTEM_MATCH
            else "INVALID fixed-system audit: systems differ"
        )
        fig.suptitle(
            f"Winnebago 10M {budget_audit_label} active-box budget ablation "
            "(whiskers=min–max)"
        )
        fig.tight_layout()
        budget_plot_path = DRIVE_RUN_ROOT / "winnebago_box_budget_10m.png"
        fig.savefig(budget_plot_path, dpi=180, bbox_inches="tight")
        GENERATED_PLOT_PATHS.append(budget_plot_path)
        plt.show()

# Scale protocols remain separate: never connect archived exact, development master,
# Manitowoc master, OAT, or paper_10m rows into one curve.
SCALE_OUTPUT_GROUPS = {
    "scale_archived_exact", "scale_development_masters", "scale_manitowoc_master"
}
scale_plot = (
    controlled_plot.loc[controlled_plot["output_group"].isin(SCALE_OUTPUT_GROUPS)].copy()
    if not controlled_plot.empty else pd.DataFrame()
)
scale_method_availability_rows = []
if not scale_plot.empty:
    for (output_group, family), family_frame in scale_plot.groupby(
        ["output_group", "dataset_family"], sort=True
    ):
        union_methods = {
            str(method) for method in family_frame["method"].dropna().astype(str)
        }
        for n_train, n_frame in family_frame.groupby("N", sort=True):
            available_methods = {
                str(method) for method in n_frame["method"].dropna().astype(str)
            }
            ordered_methods = [
                method for method in METHOD_ORDER if method in available_methods
            ] + sorted(available_methods - set(METHOD_ORDER))
            scale_method_availability_rows.append({
                "output_group": output_group,
                "dataset_family": family,
                "N": int(n_train),
                "methods": ",".join(ordered_methods),
                "method_count": len(available_methods),
                "is_subset_vs_profile_union": available_methods != union_methods,
            })
SCALE_METHOD_AVAILABILITY_PATH = DRIVE_RUN_ROOT / "scale_method_availability.csv"
pd.DataFrame(scale_method_availability_rows).to_csv(
    SCALE_METHOD_AVAILABILITY_PATH, index=False
)
scale_profiles = (
    scale_plot.groupby("output_group", sort=True)
    if not scale_plot.empty else ()
)
for output_group, profile_frame in scale_profiles:
    assert_cg_reference_one(profile_frame, context=output_group)
    profile_sources = profile_frame["source_bundle_sha256"].dropna().astype(str)
    if len(profile_sources) != len(profile_frame) or profile_sources.nunique() != 1:
        raise RuntimeError(f"{output_group} mixes or lacks source bundles.")
    families = [name for name in DATASET_ORDER if name in set(profile_frame["dataset_family"])]
    families += sorted(set(profile_frame["dataset_family"]) - set(families))
    fig, axes = plt.subplots(
        1, len(families), figsize=(5.2 * len(families), 4.6), squeeze=False, sharey=True,
    )
    for ax, family in zip(axes[0], families):
        family_frame = profile_frame.loc[profile_frame["dataset_family"].eq(family)]
        for runtime_field in (
            "device_name", "compute_capability", "timing_runtime_sha256",
        ):
            runtime_values = family_frame[runtime_field].dropna().astype(str)
            if len(runtime_values) != len(family_frame) or runtime_values.nunique() != 1:
                raise RuntimeError(
                    f"{output_group}/{family} mixes or lacks {runtime_field}; "
                    "do not join scale points measured on different GPU runtimes."
                )
        series_values = family_frame["dataset_series_id"].dropna().astype(str)
        if (
            len(series_values) != len(family_frame)
            or not bool(series_values.str.len().gt(0).all())
            or series_values.nunique() != 1
        ):
            raise RuntimeError(
                f"{output_group}/{family} mixes or lacks dataset series."
            )
        data_fields = [
            "N", "dataset_stem", "dataset_content_index_sha256",
            "dataset_metadata_sha256",
        ]
        case_data = family_frame[data_fields].drop_duplicates()
        if bool(case_data[data_fields].isna().any().any()):
            raise RuntimeError(f"{output_group}/{family} lacks dataset provenance.")
        if bool(pd.to_numeric(case_data["N"], errors="coerce").duplicated().any()):
            raise RuntimeError(
                f"{output_group}/{family} has multiple data artifacts for one N."
            )
        if output_group in {"scale_development_masters", "scale_manitowoc_master"}:
            for field in (
                "dataset_stem", "dataset_content_index_sha256", "dataset_metadata_sha256"
            ):
                if case_data[field].astype(str).nunique() != 1:
                    raise RuntimeError(
                        f"{output_group}/{family} does not reuse one master {field}."
                    )
        family_union_methods = set(family_frame["method"].dropna().astype(str))
        method_subset_notes = []
        for n_train, n_frame in family_frame.groupby("N", sort=True):
            available_methods = set(n_frame["method"].dropna().astype(str))
            if available_methods != family_union_methods:
                ordered_available = [
                    method for method in METHOD_ORDER if method in available_methods
                ] + sorted(available_methods - set(METHOD_ORDER))
                method_subset_notes.append(
                    f"{int(n_train) / 1e6:g}M: {', '.join(ordered_available)}"
                )
        for method in METHOD_ORDER:
            group = family_frame.loc[family_frame["method"].eq(method)].sort_values("N")
            if group.empty:
                continue
            if group["scientific_config_id"].nunique(dropna=False) != 1:
                raise RuntimeError(
                    f"{output_group}/{family}/{method} mixes scientific configurations."
                )
            if bool(pd.to_numeric(group["N"], errors="coerce").duplicated().any()):
                raise RuntimeError(f"{output_group}/{family}/{method} has duplicate N values.")
            x_values = pd.to_numeric(group["N"], errors="raise").to_numpy(float)
            y_values, yerr = minmax_error(
                group, "cold_speedup_median", "cold_speedup_min", "cold_speedup_max"
            )
            linestyle = "-" if np.unique(x_values).size >= 2 else "None"
            ax.errorbar(
                x_values, y_values, yerr=yerr, marker="o", ls=linestyle, capsize=3,
                color=METHOD_COLORS.get(method), label=method,
            )
        ax.set_xscale("log")
        ax.axhline(1.0, color="black", lw=1, ls="--")
        ax.set_title(family)
        ax.set_xlabel("training rows N")
        ax.grid(True, alpha=.25)
        if method_subset_notes:
            ax.text(
                0.02, 0.02,
                "method subsets at large N\n" + "\n".join(method_subset_notes),
                transform=ax.transAxes, ha="left", va="bottom", fontsize=7.5,
                bbox={"facecolor": "white", "edgecolor": "0.8", "alpha": 0.85},
            )
    axes[0][0].set_ylabel("paired cold speedup over CG")
    handles = [
        Line2D([0], [0], marker="o", color=METHOD_COLORS[m], label=m)
        for m in METHOD_ORDER if m in set(profile_frame["method"])
    ]
    fig.legend(handles=handles, ncol=min(3, len(handles)), frameon=False,
               loc="upper center", bbox_to_anchor=(0.5, 1.04))
    fig.suptitle(
        f"Controlled scale: {output_group} (median; whiskers=min–max)", y=1.11
    )
    fig.tight_layout()
    scale_plot_path = DRIVE_RUN_ROOT / f"{output_group}_cold_speedup.png"
    fig.savefig(scale_plot_path, dpi=180, bbox_inches="tight")
    GENERATED_PLOT_PATHS.append(scale_plot_path)
    plt.show()

    setup_profile = profile_frame.loc[
        profile_frame["setup_inclusive_timing_eligible"].astype(str).str.lower().eq("true")
    ].copy()
    if not setup_profile.empty:
        fig_setup, setup_axes = plt.subplots(
            1, len(families), figsize=(5.2 * len(families), 4.6),
            squeeze=False, sharey=True,
        )
        for ax_setup, family in zip(setup_axes[0], families):
            family_setup = setup_profile.loc[
                setup_profile["dataset_family"].eq(family)
            ]
            for method in METHOD_ORDER:
                group = family_setup.loc[
                    family_setup["method"].eq(method)
                ].sort_values("N")
                if group.empty:
                    continue
                x_values = pd.to_numeric(
                    group["N"], errors="raise"
                ).to_numpy(float)
                y_values = pd.to_numeric(
                    group["shared_fourier_setup_plus_method_speedup_median"],
                    errors="raise",
                ).to_numpy(float)
                ax_setup.plot(
                    x_values, y_values, marker="o",
                    ls="-" if np.unique(x_values).size >= 2 else "None",
                    color=METHOD_COLORS.get(method), label=method,
                )
            ax_setup.set_xscale("log")
            ax_setup.axhline(1.0, color="black", lw=1, ls="--")
            ax_setup.set_title(family)
            ax_setup.set_xlabel("training rows N")
            ax_setup.grid(True, alpha=.25)
        setup_axes[0][0].set_ylabel(
            "setup-inclusive median speedup over CG"
        )
        fig_setup.legend(
            handles=handles, ncol=min(3, len(handles)), frameon=False,
            loc="upper center", bbox_to_anchor=(0.5, 1.04),
        )
        fig_setup.suptitle(
            f"Controlled scale: {output_group} (shared Fourier setup included; "
            "artifact-reused setup timings excluded)", y=1.11,
        )
        fig_setup.tight_layout()
        setup_plot_path = (
            DRIVE_RUN_ROOT / f"{output_group}_setup_inclusive_speedup.png"
        )
        fig_setup.savefig(setup_plot_path, dpi=180, bbox_inches="tight")
        GENERATED_PLOT_PATHS.append(setup_plot_path)
        plt.show()
    else:
        print(
            f"No eligible setup-inclusive timing rows for {output_group}; "
            "solver-only scale plot remains valid."
        )

deprecated_plot = DRIVE_RUN_ROOT / "controlled_scale_speedup.png"
if deprecated_plot.is_file():
    print("Deprecated ambiguous plot remains on Drive but is excluded:", deprecated_plot)
print("Generated plot paths:", [str(path) for path in GENERATED_PLOT_PATHS])


# E. 最终 checkpoint、Drive 校验与可选断开

结果目录只包含配置、数据 manifest 快照、CSV/JSON 和图，不复制大数据。自动断开默认关闭；只有下格确认所有期望 case 均有 `run_complete.json`、prediction audit 严格复用 timing system/solutions，且统一索引已写入 Drive 后才允许开启。


In [ ]:
CAMPAIGN_EXECUTION_FINISHED_UTC = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
current_campaign_invocation_elapsed_seconds = (
    time.perf_counter() - CAMPAIGN_EXECUTION_STARTED_PERF_COUNTER
)
current_notebook_invocation_elapsed_seconds = (
    time.perf_counter() - NOTEBOOK_STARTED_PERF_COUNTER
)
prior_final_manifest = {}
prior_manifest_path = DRIVE_RUN_ROOT / "colab_run_manifest.json"
if prior_manifest_path.is_file():
    try:
        prior_final_manifest = json.loads(prior_manifest_path.read_text(encoding="utf-8"))
        if not isinstance(prior_final_manifest, dict):
            raise TypeError("prior final manifest is not an object")
    except (OSError, json.JSONDecodeError, TypeError):
        prior_final_manifest = {}
        print("WARNING: prior final manifest is unreadable; first-run total cannot be carried forward.")
non_smoke_jobs = [row for row in campaign_job_rows if row.get("job_id") != "plumbing_smoke"]
all_campaign_jobs_fresh = bool(
    non_smoke_jobs
    and all(
        row.get("invocation_mode") == "executed"
        and row.get("status") == "PASS"
        for row in non_smoke_jobs
    )
)
if all_campaign_jobs_fresh:
    first_run_campaign_elapsed_seconds = current_campaign_invocation_elapsed_seconds
    first_run_campaign_elapsed_source = "current invocation"
elif (
    prior_final_manifest.get("run_verified") is True
    and prior_final_manifest.get("first_run_campaign_elapsed_seconds") is not None
):
    first_run_campaign_elapsed_seconds = float(
        prior_final_manifest["first_run_campaign_elapsed_seconds"]
    )
    first_run_campaign_elapsed_source = "preserved prior final manifest"
else:
    first_run_campaign_elapsed_seconds = None
    first_run_campaign_elapsed_source = (
        "unavailable: one or more artifacts predated the timing ledger"
    )
resume_summary = {
    "executed_job_count": sum(
        row.get("invocation_mode") == "executed" for row in campaign_job_rows
    ),
    "resumed_job_count": sum(
        row.get("invocation_mode") == "resumed_existing" for row in campaign_job_rows
    ),
    "mixed_job_count": sum(
        row.get("invocation_mode") == "mixed_execute_and_resume" for row in campaign_job_rows
    ),
    "resumed_case_count": sum(
        int(row.get("resumed_case_count", 0) or 0) for row in campaign_job_rows
    ),
    "executed_case_count": sum(
        int(row.get("executed_case_count", 0) or 0) for row in campaign_job_rows
    ),
}
final_manifest = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "notebook_invocation_started_utc": NOTEBOOK_STARTED_UTC,
    "campaign_execution_started_utc": CAMPAIGN_EXECUTION_STARTED_UTC,
    "campaign_execution_finished_utc": CAMPAIGN_EXECUTION_FINISHED_UTC,
    "current_notebook_invocation_elapsed_seconds": current_notebook_invocation_elapsed_seconds,
    "current_campaign_invocation_elapsed_seconds": current_campaign_invocation_elapsed_seconds,
    "first_run_campaign_elapsed_seconds": first_run_campaign_elapsed_seconds,
    "first_run_campaign_elapsed_seconds_source": first_run_campaign_elapsed_source,
    "resume_summary": resume_summary,
    "elapsed_time_semantics": {
        "campaign_jobs.elapsed_seconds": (
            "wall time of the current invocation; for resumed_existing it is only "
            "resume validation and must not be reported as first-run experiment time"
        ),
        "campaign_jobs.first_run_elapsed_seconds": (
            "exact fresh suite wall time when observed, otherwise preserved or null; "
            "never a per-method timing claim"
        ),
        "paper_method_timing": (
            "use matched_summary.csv paired selection/build+solve columns, not campaign wall time"
        ),
    },
    "git_sha": GIT_SHA,
    "runtime": runtime_info,
    "data_manifest": str(DATA_MANIFEST),
    "data_manifest_sha256": DATA_MANIFEST_SHA256,
    "data_manifest_snapshot": str(DATA_MANIFEST_SNAPSHOT),
    "data_manifest_snapshot_sha256": hashlib.sha256(
        DATA_MANIFEST_SNAPSHOT.read_bytes()
    ).hexdigest(),
    "selected_data_bundles": DATA_BUNDLES,
    "run_tag": RUN_TAG,
    "active_sizes": [int(n) for n in ACTIVE_SIZES],
    "independent_manitowoc_300m_status": (
        "requested" if RUN_MANITOWOC_SCALE and 300_000_000 in ACTIVE_SIZES
        else "pending_data_generation_and_prefix_verification"
    ),
    "real_data_300m_route": (
        "Winnebago archived-exact; independent Manitowoc is currently 10M only"
    ),
    "legacy_groups": list(RUN_LEGACY_GROUPS),
    "controlled_profiles": selected_profiles,
    "campaign_jobs_csv": str(CAMPAIGN_JOBS_CSV),
    "campaign_jobs_json": str(CAMPAIGN_JOBS_JSON),
    "campaign_jobs": campaign_job_rows,
    "controlled_case_count": int(len(controlled_artifact_audit)),
    "expected_controlled_case_count": int(expected_controlled_case_count),
    "selected_controlled_cases": [
        {
            **{key: value for key, value in record.items() if key != "run_dir"},
            "run_dir": str(record["run_dir"]),
        }
        for record in selected_case_records
    ],
    "ignored_stale_controlled_run_count": int(len(ignored_stale_dirs)),
    "all_controlled_artifacts_pass": (
        None if expected_controlled_case_count == 0 else bool(
            len(controlled_artifact_audit) == expected_controlled_case_count
            and controlled_artifact_audit["status"].eq("PASS").all()
        )
    ),
    "unified_index": str(INDEX_PATH),
    "selected_controlled_index": str(SELECTED_CONTROLLED_INDEX_PATH),
    "controlled_artifact_audit": str(CONTROLLED_AUDIT_PATH),
    "controlled_ineligible_rows": str(INELIGIBLE_INDEX_PATH),
    "report_ingest_warnings": str(REPORT_INGEST_WARNINGS_PATH),
    "prediction_artifact_audit": str(PREDICTION_ARTIFACT_AUDIT_PATH),
    "prediction_accuracy_summary": str(PREDICTION_ACCURACY_SUMMARY_PATH),
    "prediction_accuracy_plot": str(DRIVE_RUN_ROOT / "prediction_accuracy_vs_cg.png"),
    "scale_method_availability": str(SCALE_METHOD_AVAILABILITY_PATH),
    "generated_plots": [str(path) for path in GENERATED_PLOT_PATHS],
    "expected_prediction_case_count": int(expected_prediction_case_count),
    "box_budget_fixed_system_match": BOX_BUDGET_SYSTEM_MATCH,
}
legacy_complete = all(
    (DRIVE_RUN_ROOT / "legacy_archived_pipeline" / group / "_SUCCESS.json").is_file()
    for group in RUN_LEGACY_GROUPS
)
expected_selected_controlled_pairs = set()
selected_config_method_schema_valid = True
for record in selected_case_records:
    config_path = Path(record["run_dir"]) / "experiment_config.json"
    try:
        selected_config = json.loads(config_path.read_text(encoding="utf-8"))
        if not isinstance(selected_config, dict):
            raise TypeError("experiment_config is not an object")
        methods = selected_config.get("methods")
        if (
            not isinstance(methods, list)
            or not methods
            or len({str(method) for method in methods}) != len(methods)
        ):
            raise ValueError("invalid experiment_config.methods")
    except (OSError, json.JSONDecodeError, TypeError, ValueError):
        selected_config_method_schema_valid = False
        continue
    run_dir_key = str(Path(record["run_dir"]).resolve())
    expected_selected_controlled_pairs.update(
        (run_dir_key, str(method)) for method in methods
    )
observed_selected_controlled_pairs = (
    set(
        zip(
            selected_controlled["run_dir"].astype(str),
            selected_controlled["method"].astype(str),
        )
    )
    if not selected_controlled.empty else set()
)
selected_controlled_summary_complete = bool(
    selected_config_method_schema_valid
    and observed_selected_controlled_pairs
    == expected_selected_controlled_pairs
    and len(selected_controlled) == len(expected_selected_controlled_pairs)
    and (
        selected_controlled.empty
        or selected_controlled["claim_eligible"].eq(True).all()
    )
)
final_manifest["selected_controlled_summary_complete"] = (
    selected_controlled_summary_complete
)
final_manifest["expected_selected_controlled_method_row_count"] = int(
    len(expected_selected_controlled_pairs)
)
controlled_complete = (
    expected_controlled_case_count == 0
    or (
        len(controlled_artifact_audit) == expected_controlled_case_count
        and controlled_artifact_audit["status"].eq("PASS").all()
        and selected_controlled_summary_complete
    )
)
prediction_validation_frame = pd.DataFrame(prediction_validation_rows)
expected_prediction_summary_pairs = {
    (str(row.get("case_id")), method)
    for row in prediction_validation_rows
    for method in str(row.get("required_methods", "")).split(",")
    if method
}
observed_prediction_summary_pairs = (
    set(
        zip(
            selected_prediction["case_id"].astype(str),
            selected_prediction["method"].astype(str),
        )
    )
    if not selected_prediction.empty else set()
)
prediction_summary_complete = bool(
    not RUN_PREDICTION_AUDIT
    or (
        expected_prediction_case_count > 0
        and expected_prediction_summary_pairs
        and observed_prediction_summary_pairs
        == expected_prediction_summary_pairs
        and len(selected_prediction)
        == len(expected_prediction_summary_pairs)
        and selected_prediction["prediction_claim_eligible"].eq(True).all()
    )
)
prediction_artifacts_pass = bool(
    len(prediction_validation_frame) == expected_prediction_case_count
    and not prediction_validation_frame.empty
    and prediction_validation_frame["status"].astype(str).str.startswith("PASS").all()
    and prediction_validation_frame["audit_pass"].eq(True).all()
    and prediction_validation_frame["exact_timing_system_match"].eq(True).all()
    and prediction_validation_frame["timing_solutions_reused"].eq(True).all()
    and prediction_validation_frame["timing_solution_hashes_verified"].eq(True).all()
    and prediction_validation_frame["zero_audit_solves"].eq(True).all()
    and prediction_summary_complete
)
final_manifest["prediction_summary_complete"] = prediction_summary_complete
final_manifest["prediction_artifacts_pass"] = prediction_artifacts_pass
final_manifest["prediction_audits_verified"] = prediction_artifacts_pass
prediction_complete = (
    not RUN_PREDICTION_AUDIT
    or (
        expected_prediction_case_count > 0
        and len(prediction_outputs) == expected_prediction_case_count
        and all(
            (Path(path) / PREDICTION_AUDIT_JSON_FILENAME).is_file()
            and (Path(path) / PREDICTION_AUDIT_CSV_FILENAME).is_file()
            and (Path(path) / PREDICTION_AUDIT_COMPLETION_FILENAME).is_file()
            for path in prediction_outputs
        )
        and prediction_artifacts_pass
    )
)
selected_output_groups = (
    set(selected_controlled["output_group"].astype(str))
    if not selected_controlled.empty else set()
)
expected_plot_paths = []
if "paper_10m" in selected_output_groups:
    expected_plot_paths.extend([
        DRIVE_RUN_ROOT / "controlled_10m_method_speedup.png",
        DRIVE_RUN_ROOT / "controlled_10m_speed_memory_pareto.png",
    ])
if "winnebago_oat_n10m" in selected_output_groups:
    expected_plot_paths.append(
        DRIVE_RUN_ROOT / "winnebago_oat_10m_speed_memory.png"
    )
if "winnebago_box_budget_n10m" in selected_output_groups:
    expected_plot_paths.append(
        DRIVE_RUN_ROOT / "winnebago_box_budget_10m.png"
    )
selected_scale_groups = sorted(
    selected_output_groups.intersection(SCALE_OUTPUT_GROUPS)
)
setup_timing_coverage_complete = True
for output_group in selected_scale_groups:
    expected_plot_paths.extend([
        DRIVE_RUN_ROOT / f"{output_group}_cold_speedup.png",
        DRIVE_RUN_ROOT / f"{output_group}_setup_inclusive_speedup.png",
    ])
    eligible_setup_rows = controlled_plot.loc[
        controlled_plot["output_group"].astype(str).eq(output_group)
        & controlled_plot["setup_inclusive_timing_eligible"].astype(str).str.lower().eq("true")
    ]
    expected_setup_run_dirs = {
        str(Path(record["run_dir"]).resolve())
        for record in selected_case_records
        if str(record["output_group"]) == output_group
    }
    observed_setup_run_dirs = set(
        eligible_setup_rows["run_dir"].astype(str)
    )
    if observed_setup_run_dirs != expected_setup_run_dirs:
        setup_timing_coverage_complete = False
if RUN_PREDICTION_AUDIT:
    expected_plot_paths.append(
        DRIVE_RUN_ROOT / "prediction_accuracy_vs_cg.png"
    )
generated_plot_keys = {
    str(Path(path).resolve()) for path in GENERATED_PLOT_PATHS
}
plot_artifacts_complete = bool(
    setup_timing_coverage_complete
    and all(
        path.is_file() and str(path.resolve()) in generated_plot_keys
        for path in expected_plot_paths
    )
)
final_manifest["expected_plot_artifacts"] = [
    str(path) for path in expected_plot_paths
]
final_manifest["setup_timing_coverage_complete"] = (
    setup_timing_coverage_complete
)
final_manifest["plot_artifacts_complete"] = plot_artifacts_complete
workload_requested = bool(
    RUN_LEGACY_GROUPS or selected_profiles or extra_suites or RUN_PREDICTION_AUDIT
)
mandatory_jobs = [row for row in campaign_job_rows if bool(row.get("mandatory", True))]
campaign_complete = bool(
    mandatory_jobs and all(str(row.get("status")) == "PASS" for row in mandatory_jobs)
)
run_verified = bool(
    workload_requested and legacy_complete and controlled_complete
    and prediction_complete and campaign_complete and INDEX_PATH.is_file()
    and plot_artifacts_complete
    and (BOX_BUDGET_SYSTEM_MATCH is not False)
)
if not run_verified and first_run_campaign_elapsed_source == "current invocation":
    first_run_campaign_elapsed_seconds = None
    first_run_campaign_elapsed_source = (
        "unavailable: fresh campaign did not pass final verification"
    )
    final_manifest["first_run_campaign_elapsed_seconds"] = None
    final_manifest["first_run_campaign_elapsed_seconds_source"] = (
        first_run_campaign_elapsed_source
    )
final_manifest["campaign_complete"] = campaign_complete
final_manifest["run_verified"] = run_verified
FINAL_MANIFEST_PATH = DRIVE_RUN_ROOT / "colab_run_manifest.json"
final_manifest_partial = FINAL_MANIFEST_PATH.with_suffix(".json.partial")
final_manifest_partial.write_text(
    json.dumps(final_manifest, indent=2), encoding="utf-8"
)
final_manifest_partial.replace(FINAL_MANIFEST_PATH)
print(json.dumps(final_manifest, indent=2))
if run_verified:
    print("ONE-CLICK CAMPAIGN VERIFIED: all mandatory jobs passed.")
else:
    print(
        "ONE-CLICK CAMPAIGN COMPLETED WITH FAILURES/SKIPS. "
        "See campaign_jobs.csv and controlled_artifact_audit.csv; completed results remain usable."
    )

if DISCONNECT_RUNTIME_WHEN_VERIFIED:
    if not run_verified or not FINAL_MANIFEST_PATH.is_file():
        raise RuntimeError("Selected workload is not fully verified; refusing to disconnect.")
    if IS_COLAB:
        from google.colab import runtime
        runtime.unassign()


## 运行后检查清单

1. 每个 controlled case 的 `system_manifest.json`：`system_unchanged=true`、`nufft_stage=cufinufft`，且 weights/Gf/solve-RHS/storage-RHS/λ 哈希完整。
2. 每个正式方法：5/5 repeats 满足 independently recomputed residual (<10^{-7})。
3. 300M 若只跑 core methods，表中明确列出方法集合；不要与另一次 invocation 的随机方法拼成 paired table。
4. Legacy、controlled、prediction audit 保持不同 `protocol_family`。
5. Synthetic 表中明确区分 archived noise=.3 `_ntrainN` 与 development noise=.02 `_nN`。
6. Prediction 必须同时有 JSON、CSV 和 completion manifest，严格使用 cuFINUFFT，并保持 audit solve count=0。
7. 记录完整 timing-runtime SHA（GPU、compute capability、CuPy/CUDA/NUFFT）、Git SHA、数据 SHA、实际 (M)、box size/rank 和 timing scope；artifact 恢复的 setup 时间不得进入 setup-inclusive 图。
